# 8특성 입력과 9특성 출력 Seq2Seq 실험

원본 최종 수정 기록: 2025-12-11T23:58:23+09:00.

파일명의 번호는 파이프라인 순서이며 버전 번호가 아닙니다. 데이터·모델 경로와 실행 환경을 설정한 뒤 필요한 단계를 실행하세요.


In [ ]:
import pandas as pd
import numpy as np
import os
import pickle
from sklearn.preprocessing import StandardScaler, LabelEncoder
from typing import Tuple, List, Dict

# 설정
INPUT_FILE = './data/kaggle-dataset/vct_combined_all_years.csv'
OUTPUT_DIR = './data/seq2seq_data'
SEQUENCE_LENGTH = 12  # 전반 12라운드
FUTURE_STEPS = 12     # 후반 12라운드

def extract_features():
    """
    Seq2Seq 모델 학습을 위한 특성 추출
    
    논문 Table 5 기반:
    - Round Index: 순서 정보
    - My Score / Opp Score: 누적 점수
    - My Bank: 팀 자금
    - My Loadout Value: 장비 투자액
    - Buy Type: 구매 유형 (Eco, Semi, Force, Full)
    - My Kills / Opp Kills: 킬 수
    - Round Outcome: 라운드 승패 (0: Loss, 1: Win)
    """
    
    print("="*80)
    print("Step 2: Seq2Seq 모델 특성 추출")
    print("="*80)
    
    # 1. 데이터 로드
    print("\n[1/6] 통합 데이터 로드 중...")
    try:
        df = pd.read_csv(INPUT_FILE)
        print(f"✅ 로드 완료: {len(df):,}개 행, {df['Game ID'].nunique()}개 게임")
    except FileNotFoundError:
        print(f"❌ 파일 없음: {INPUT_FILE}")
        return False
    
    # 2. 기본 전처리
    print("\n[2/6] 기본 전처리 중...")
     # 2. 기본 전처리
    print("\n[2/6] 기본 전처리 중...")

    # 숫자형 컬럼 문자열 처리 (예: '0.3k' → 300)
    def parse_k_value(x):
        if isinstance(x, str):
            x = x.strip()
            if x.endswith('k'):
                try:
                    return float(x[:-1]) * 1000.0
                except ValueError:
                    return np.nan
            try:
                return float(x.replace(',', ''))
            except ValueError:
                return np.nan
        return x

    num_cols = []
    for col in ['Loadout Value', 'Remaining Credits', 'My_Kills', 'Opp_Kills', 'Cumulative Score']:
        if col in df.columns:
            num_cols.append(col)
            df[col] = df[col].apply(parse_k_value)

    print(f"   숫자형 변환 대상 컬럼: {num_cols}")
    
    # 결측치 확인
    print(f"\n결측치 확인:")
    missing = df.isnull().sum()
    missing_cols = missing[missing > 0]
    if len(missing_cols) > 0:
        print(f"  결측치 있는 컬럼:")
        for col, count in missing_cols.items():
            print(f"    {col}: {count:,}개 ({count/len(df)*100:.2f}%)")
    else:
        print(f"  ✅ 결측치 없음")
    
    # Outcome 인코딩 (Win=1, Loss=0)
    df['Outcome_Encoded'] = (df['Outcome'] == 'Win').astype(int)
    print(f"\nOutcome 인코딩:")
    print(f"  Win: 1, Loss: 0")
    print(f"  분포: Win {(df['Outcome_Encoded']==1).sum():,} ({(df['Outcome_Encoded']==1).sum()/len(df)*100:.1f}%)")
    
    # Buy Type 인코딩
    buy_type_mapping = {'Eco': 0, 'Semi': 1, 'Force': 2, 'Full': 3}
    # Buy Type 컬럼이 있으면 인코딩, 없으면 스킵
    if 'Type' in df.columns:
        df['Buy Type Encoded'] = df['Type'].map(buy_type_mapping)
        df['Buy Type Encoded'] = df['Buy Type Encoded'].fillna(0)  # 알 수 없는 타입은 0
        print(f"\nBuy Type 인코딩: {buy_type_mapping}")
    else:
        print(f"\n⚠️  Buy Type 컬럼 없음 - Type 컬럼으로 대체")
        df['Buy Type Encoded'] = 0
    
    # 3. 게임별 시퀀스 생성
    print("\n[3/6] 게임별 시퀀스 생성 중...")
    
    sequences = []
    invalid_games = []
    
    for game_id, game_data in df.groupby('Game ID'):
        game_data = game_data.sort_values('Round Number').reset_index(drop=True)
        
        # 팀별로 처리
        for team in game_data['Team'].unique():
            team_seq = game_data[game_data['Team'] == team].reset_index(drop=True)
            
            # 라운드 수 확인
            if len(team_seq) < SEQUENCE_LENGTH + FUTURE_STEPS:
                invalid_games.append({
                    'game_id': game_id,
                    'team': team,
                    'rounds': len(team_seq),
                    'required': SEQUENCE_LENGTH + FUTURE_STEPS
                })
                continue
            
            # 상대팀 데이터 추출
            opp_team = game_data[game_data['Team'] != team].reset_index(drop=True)
            
            # 시퀀스 생성 (겹치지 않게)
            for start_idx in range(0, len(team_seq) - SEQUENCE_LENGTH - FUTURE_STEPS + 1):
                # 입력 시퀀스 (전반 12라운드)
                input_seq_indices = range(start_idx, start_idx + SEQUENCE_LENGTH)
                input_seq = team_seq.iloc[input_seq_indices].reset_index(drop=True)
                
                # 출력 시퀀스 (후반 12라운드)
                output_seq_indices = range(start_idx + SEQUENCE_LENGTH, 
                                          start_idx + SEQUENCE_LENGTH + FUTURE_STEPS)
                output_seq = team_seq.iloc[output_seq_indices].reset_index(drop=True)
                
                # 상대팀 데이터
                opp_input_seq = opp_team.iloc[input_seq_indices].reset_index(drop=True)
                opp_output_seq = opp_team.iloc[output_seq_indices].reset_index(drop=True)
                
                sequences.append({
                    'game_id': game_id,
                    'team': team,
                    'input_seq': input_seq,
                    'output_seq': output_seq,
                    'opp_input_seq': opp_input_seq,
                    'opp_output_seq': opp_output_seq
                })
    
    print(f"✅ 시퀀스 생성 완료: {len(sequences)}개")
    print(f"⚠️  불완전한 게임-팀 조합: {len(invalid_games)}개")
    if len(invalid_games) > 0 and len(invalid_games) <= 10:
        for item in invalid_games[:10]:
            print(f"    Game {item['game_id']}, Team {item['team']}: {item['rounds']}라운드 (필요: {item['required']})")
    
    # 4. 특성 벡터 변환
    print("\n[4/6] 특성 벡터 변환 중...")
    
    X_sequences = []
    Y_sequences = []
    
    for seq_data in sequences:
        input_seq = seq_data['input_seq']
        output_seq = seq_data['output_seq']
        opp_input_seq = seq_data['opp_input_seq']
        opp_output_seq = seq_data['opp_output_seq']
        
        # 입력 특성 추출 (SEQUENCE_LENGTH, 8)
        X = []
        for idx in range(len(input_seq)):
            features = [
                idx,  # Round Index (0~11)
                input_seq['Cumulative Score'].iloc[idx],  # My Score
                opp_input_seq['Cumulative Score'].iloc[idx],  # Opp Score
                input_seq['Remaining Credits'].iloc[idx] if 'Remaining Credits' in input_seq.columns else 0,  # My Bank
                input_seq['Loadout Value'].iloc[idx] if 'Loadout Value' in input_seq.columns else 0,  # Loadout Value
                input_seq['Buy Type Encoded'].iloc[idx],  # Buy Type
                input_seq['My_Kills'].iloc[idx],  # My Kills
                input_seq['Opp_Kills'].iloc[idx],  # Opp Kills
            ]
            X.append(features)
        
        X_sequences.append(np.array(X, dtype=np.float32))
        
        # 출력 특성 추출 (FUTURE_STEPS, 8)
        Y = []
        for idx in range(len(output_seq)):
            features = [
                SEQUENCE_LENGTH + idx,  # Round Index (12~23)
                output_seq['Cumulative Score'].iloc[idx],  # My Score
                opp_output_seq['Cumulative Score'].iloc[idx],  # Opp Score
                output_seq['Remaining Credits'].iloc[idx] if 'Remaining Credits' in output_seq.columns else 0,  # My Bank
                output_seq['Loadout Value'].iloc[idx] if 'Loadout Value' in output_seq.columns else 0,  # Loadout Value
                output_seq['Buy Type Encoded'].iloc[idx],  # Buy Type
                output_seq['My_Kills'].iloc[idx],  # My Kills
                output_seq['Opp_Kills'].iloc[idx],  # Opp Kills
                output_seq['Outcome_Encoded'].iloc[idx],  # Round Outcome (타겟)
            ]
            Y.append(features)
        
        Y_sequences.append(np.array(Y, dtype=np.float32))
    
    X_sequences = np.array(X_sequences)
    Y_sequences = np.array(Y_sequences)
    
    print(f"✅ 특성 벡터 변환 완료")
    print(f"   X 형태: {X_sequences.shape} (시퀀스, 라운드, 특성)")
    print(f"   Y 형태: {Y_sequences.shape} (시퀀스, 라운드, 특성+타겟)")
    
    # 5. 데이터 정규화
    print("\n[5/6] 데이터 정규화 중...")
    
    # X 정규화 (모든 시퀀스)
    X_flat = X_sequences.reshape(-1, X_sequences.shape[-1])
    scaler_X = StandardScaler()
    X_flat_scaled = scaler_X.fit_transform(X_flat)
    X_sequences_scaled = X_flat_scaled.reshape(X_sequences.shape)
    
    # Y 정규화 (마지막 컬럼 제외 - Round Outcome)
    Y_flat = Y_sequences.reshape(-1, Y_sequences.shape[-1])
    scaler_Y = StandardScaler()
    Y_flat_scaled = scaler_Y.fit_transform(Y_flat[:, :-1])  # Outcome 제외
    Y_flat_scaled_with_outcome = np.hstack([Y_flat_scaled, Y_flat[:, -1:]])  # Outcome 다시 추가
    Y_sequences_scaled = Y_flat_scaled_with_outcome.reshape(Y_sequences.shape)
    
    print(f"✅ 정규화 완료")
    print(f"   X 정규화: mean={X_sequences_scaled.mean():.4f}, std={X_sequences_scaled.std():.4f}")
    print(f"   Y 정규화: mean={Y_sequences_scaled[:,:,:-1].mean():.4f}, std={Y_sequences_scaled[:,:,:-1].std():.4f}")
    
    # 6. 저장
    print("\n[6/6] 데이터 저장 중...")
    
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    
    # NumPy 파일로 저장
    np.save(f"{OUTPUT_DIR}/X_sequences.npy", X_sequences_scaled)
    np.save(f"{OUTPUT_DIR}/Y_sequences.npy", Y_sequences_scaled)
    
    # 스케일러 저장 (나중에 예측 시 사용)
    with open(f"{OUTPUT_DIR}/scaler_X.pkl", 'wb') as f:
        pickle.dump(scaler_X, f)
    with open(f"{OUTPUT_DIR}/scaler_Y.pkl", 'wb') as f:
        pickle.dump(scaler_Y, f)
    
    # 메타데이터 저장
    metadata = {
        'total_sequences': len(sequences),
        'sequence_length': SEQUENCE_LENGTH,
        'future_steps': FUTURE_STEPS,
        'feature_names': [
            'Round Index', 'My Score', 'Opp Score', 'My Bank', 
            'Loadout Value', 'Buy Type', 'My Kills', 'Opp Kills'
        ],
        'target_names': [
            'Round Index', 'My Score', 'Opp Score', 'My Bank', 
            'Loadout Value', 'Buy Type', 'My Kills', 'Opp Kills', 'Round Outcome'
        ],
        'X_shape': X_sequences_scaled.shape,
        'Y_shape': Y_sequences_scaled.shape,
    }
    
    with open(f"{OUTPUT_DIR}/metadata.pkl", 'wb') as f:
        pickle.dump(metadata, f)
    
    print(f"✅ 저장 완료: {OUTPUT_DIR}/")
    print(f"   - X_sequences.npy")
    print(f"   - Y_sequences.npy")
    print(f"   - scaler_X.pkl")
    print(f"   - scaler_Y.pkl")
    print(f"   - metadata.pkl")
    
    # 최종 요약
    print(f"\n{'='*80}")
    print("✅ 특성 추출 완료!")
    print(f"{'='*80}")
    print(f"\n데이터 통계:")
    print(f"   총 시퀀스: {len(sequences):,}개")
    print(f"   X 형태: {X_sequences_scaled.shape}")
    print(f"   Y 형태: {Y_sequences_scaled.shape}")
    print(f"   입력 특성 수: {X_sequences_scaled.shape[-1]}")
    print(f"   출력 특성 수: {Y_sequences_scaled.shape[-1]}")
    
    print(f"\n특성 설명:")
    print(f"   1. Round Index: 라운드 순서 (0~23)")
    print(f"   2. My Score: 누적 점수")
    print(f"   3. Opp Score: 상대 누적 점수")
    print(f"   4. My Bank: 팀 자금")
    print(f"   5. Loadout Value: 장비 투자액")
    print(f"   6. Buy Type: 구매 유형 (0=Eco, 1=Semi, 2=Force, 3=Full)")
    print(f"   7. My Kills: 킬 수")
    print(f"   8. Opp Kills: 상대 킬 수")
    print(f"   9. Round Outcome: 라운드 승패 (Y만, 0=Loss, 1=Win)")
    
    print(f"\n다음 단계:")
    print(f"  Step 3: Seq2Seq LSTM 모델 구축 및 학습")
    print(f"  - Encoder-Decoder LSTM")
    print(f"  - Train/Test 분할 (80/20)")
    print(f"  - 모델 학습 및 평가")
    
    return True

if __name__ == "__main__":
    extract_features()

In [ ]:
import numpy as np
import pandas as pd
import os
import pickle
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.optimizers import Adam
import warnings
warnings.filterwarnings('ignore')

# 설정
DATA_DIR = './data/seq2seq_data'
MODEL_DIR = './models/seq2seq_valorant'
BATCH_SIZE = 32
EPOCHS = 100
VALIDATION_SPLIT = 0.2
TEST_SIZE = 0.2
SEED = 42

np.random.seed(SEED)
tf.random.set_seed(SEED)

def build_seq2seq_model(sequence_length, future_steps, n_features, n_encoder_units=128, n_decoder_units=128):
    """
    Seq2Seq LSTM 모델 구축 (Encoder-Decoder 구조)
    
    논문 기반:
    - Encoder: 전반 12라운드 처리
    - Decoder: 후반 12라운드 생성
    """
    
    # Encoder 입력
    encoder_inputs = keras.Input(shape=(sequence_length, n_features), name='encoder_input')
    
    # Encoder LSTM (양방향)
    encoder_lstm = layers.Bidirectional(
        layers.LSTM(n_encoder_units, return_state=True, name='encoder_lstm')
    )
    encoder_outputs, forward_h, forward_c, backward_h, backward_c = encoder_lstm(encoder_inputs)
    
    # 상태 통합
    state_h = layers.Concatenate()([forward_h, backward_h])
    state_c = layers.Concatenate()([forward_c, backward_c])
    
    # Decoder 입력
    decoder_inputs = keras.Input(shape=(future_steps, n_features), name='decoder_input')
    
    # Decoder LSTM
    decoder_lstm = layers.LSTM(
        n_encoder_units * 2,  # Bidirectional 때문에 2배
        return_sequences=True,
        name='decoder_lstm'
    )
    decoder_outputs = decoder_lstm(decoder_inputs, initial_state=[state_h, state_c])
    
    # Dense 레이어
    decoder_dense = layers.Dense(64, activation='relu', name='decoder_dense1')(decoder_outputs)
    decoder_dropout = layers.Dropout(0.2)(decoder_dense)
    
    # 출력 레이어 (특성 9개: 8개 특성 + 1개 타겟)
    outputs = layers.Dense(9, name='output_layer')(decoder_dropout)
    
    # 모델 생성
    model = Model([encoder_inputs, decoder_inputs], outputs, name='Seq2Seq_LSTM')
    
    return model

def load_data():
    """특성 추출된 데이터 로드"""
    
    print("\n"+"="*80)
    print("Step 3: Seq2Seq LSTM 모델 구축 및 학습")
    print("="*80)
    
    print("\n[1/5] 데이터 로드 중...")
    
    X = np.load(f'{DATA_DIR}/X_sequences.npy')
    Y = np.load(f'{DATA_DIR}/Y_sequences.npy')
    
    with open(f'{DATA_DIR}/metadata.pkl', 'rb') as f:
        metadata = pickle.load(f)
    
    print(f"✅ 데이터 로드 완료")
    print(f"   X 형태: {X.shape}")
    print(f"   Y 형태: {Y.shape}")
    print(f"   총 시퀀스: {X.shape[0]:,}개")
    print(f"   입력 시퀀스 길이: {X.shape[1]}")
    print(f"   출력 시퀀스 길이: {Y.shape[1]}")
    print(f"   입력 특성: {X.shape[-1]}개")
    print(f"   출력 특성: {Y.shape[-1]}개")
    
    return X, Y, metadata

def create_train_test_split(X, Y, test_size=0.2, val_size=0.2):
    """Train/Validation/Test 분할"""
    
    print("\n[2/5] Train/Validation/Test 분할 중...")
    
    # 전체 시퀀스를 입력/출력으로 분리
    # Encoder input (X), Decoder input (Y의 처음 8개 특성), Target (Y의 마지막 특성 포함)
    decoder_inputs = Y[:, :, :-1]  # 처음 8개 특성 (Cumulative Score 포함)
    targets = Y  # 모든 9개 특성 (타겟 포함)
    
    # Train/Test 분할
    indices = np.arange(X.shape[0])
    train_idx, test_idx = train_test_split(
        indices, 
        test_size=test_size, 
        random_state=SEED
    )
    
    # Train/Val 분할
    train_idx, val_idx = train_test_split(
        train_idx,
        test_size=val_size / (1 - test_size),
        random_state=SEED
    )
    
    X_train = X[train_idx]
    X_val = X[val_idx]
    X_test = X[test_idx]
    
    decoder_train = decoder_inputs[train_idx]
    decoder_val = decoder_inputs[val_idx]
    decoder_test = decoder_inputs[test_idx]
    
    target_train = targets[train_idx]
    target_val = targets[val_idx]
    target_test = targets[test_idx]
    
    print(f"✅ 분할 완료")
    print(f"   Train: {len(train_idx):,}개 ({len(train_idx)/len(indices)*100:.1f}%)")
    print(f"   Val:   {len(val_idx):,}개 ({len(val_idx)/len(indices)*100:.1f}%)")
    print(f"   Test:  {len(test_idx):,}개 ({len(test_idx)/len(indices)*100:.1f}%)")
    
    return (X_train, decoder_train, target_train), \
           (X_val, decoder_val, target_val), \
           (X_test, decoder_test, target_test)

def train_model():
    """Seq2Seq LSTM 모델 학습"""
    
    print("\n"+"="*80)
    print("Step 3: Seq2Seq LSTM 모델 구축 및 학습")
    print("="*80)
    
    # 1. 데이터 로드
    X, Y, metadata = load_data()
    sequence_length = metadata['sequence_length']
    future_steps = metadata['future_steps']
    n_features = X.shape[-1]
    
    # 2. Train/Val/Test 분할
    train_data, val_data, test_data = create_train_test_split(X, Y)
    X_train, decoder_train, target_train = train_data
    X_val, decoder_val, target_val = val_data
    X_test, decoder_test, target_test = test_data
    
    # 3. 모델 구축
    print("\n[3/5] 모델 구축 중...")
    
    model = build_seq2seq_model(
        sequence_length=sequence_length,
        future_steps=future_steps,
        n_features=n_features,
        n_encoder_units=128,
        n_decoder_units=128
    )
    
    print(f"✅ 모델 구축 완료")
    print(f"\n모델 요약:")
    model.summary()
    
    # 4. 모델 컴파일
    print("\n[4/5] 모델 컴파일 및 학습 중...")
    
    model.compile(
        optimizer=Adam(learning_rate=0.001),
        loss='mse',
        metrics=['mae', 'mape']
    )
    
    # Callback 설정
    callbacks = [
        EarlyStopping(
            monitor='val_loss',
            patience=15,
            restore_best_weights=True,
            verbose=1
        ),
        ReduceLROnPlateau(
            monitor='val_loss',
            factor=0.5,
            patience=5,
            min_lr=1e-6,
            verbose=1
        ),
        ModelCheckpoint(
            f'{MODEL_DIR}/best_model.h5',
            monitor='val_loss',
            save_best_only=True,
            verbose=0
        )
    ]
    
    # 모델 학습
    print(f"\n학습 시작...")
    print(f"  Batch size: {BATCH_SIZE}")
    print(f"  Epochs: {EPOCHS}")
    print(f"  Early stopping patience: 15")
    
    history = model.fit(
        [X_train, decoder_train],
        target_train,
        batch_size=BATCH_SIZE,
        epochs=EPOCHS,
        validation_data=([X_val, decoder_val], target_val),
        callbacks=callbacks,
        verbose=1
    )
    
    print(f"\n✅ 학습 완료")
    
    # 5. 모델 평가
    print("\n[5/5] 모델 평가 중...")
    
    train_loss, train_mae, train_mape = model.evaluate(
        [X_train, decoder_train], 
        target_train, 
        verbose=0
    )
    
    val_loss, val_mae, val_mape = model.evaluate(
        [X_val, decoder_val], 
        target_val, 
        verbose=0
    )
    
    test_loss, test_mae, test_mape = model.evaluate(
        [X_test, decoder_test], 
        target_test, 
        verbose=0
    )
    
    print(f"✅ 평가 완료")
    print(f"\n성능 지표:")
    print(f"  Train - Loss: {train_loss:.6f}, MAE: {train_mae:.6f}, MAPE: {train_mape:.6f}%")
    print(f"  Val   - Loss: {val_loss:.6f}, MAE: {val_mae:.6f}, MAPE: {val_mape:.6f}%")
    print(f"  Test  - Loss: {test_loss:.6f}, MAE: {test_mae:.6f}, MAPE: {test_mape:.6f}%")
    
    # 최종 요약
    print(f"\n{'='*80}")
    print("✅ 모델 구축 및 학습 완료!")
    print(f"{'='*80}")
    
    print(f"\n저장 파일:")
    print(f"  모델: {MODEL_DIR}/best_model.h5")
    print(f"  히스토리: {MODEL_DIR}/training_history.pkl")
    print(f"  메타데이터: {MODEL_DIR}/model_metadata.pkl")
    
    print(f"\n모델 구조:")
    print(f"  Encoder: Bidirectional LSTM (128 units)")
    print(f"  Decoder: LSTM (256 units) → Dense (64) → Dense (9)")
    print(f"  입력: [Encoder (12라운드), Decoder (12라운드)]")
    print(f"  출력: 12라운드 × 9 특성 (Score, Kills, Outcome 등)")
    
    print(f"\n입력 특성 (8개):")
    print(f"  1. Round Index")
    print(f"  2. My Score (누적 점수)")
    print(f"  3. Opp Score (상대 누적 점수)")
    print(f"  4. My Bank (팀 자금)")
    print(f"  5. Loadout Value (장비가치)")
    print(f"  6. Buy Type (구매유형)")
    print(f"  7. My Kills (킬수)")
    print(f"  8. Opp Kills (상대 킬수)")
    
    print(f"\n출력 특성 (9개, 동일 + 타겟):")
    print(f"  + Round Outcome (라운드 승패) - 이진 분류 타겟")
    
    print(f"\n다음 단계:")
    print(f"  Step 4: 예측 및 평가")
    print(f"  - 테스트 셋에서 예측")
    print(f"  - 라운드 승패 정확도 계산")
    print(f"  - 시각화 및 분석")
    
    # 모델 저장
    os.makedirs(MODEL_DIR, exist_ok=True)
    model.save(f'{MODEL_DIR}/seq2seq_model.h5')
    
    # 히스토리 저장
    with open(f'{MODEL_DIR}/training_history.pkl', 'wb') as f:
        pickle.dump(history.history, f)
    
    # 메타데이터 저장
    model_meta = {
        'test_loss': test_loss,
        'test_mae': test_mae,
        'test_mape': test_mape,
        'val_loss': val_loss,
        'val_mae': val_mae,
        'val_mape': val_mape,
        'train_loss': train_loss,
        'train_mae': train_mae,
        'train_mape': train_mape,
        'n_train_samples': len(X_train),
        'n_val_samples': len(X_val),
        'n_test_samples': len(X_test),
    }
    
    with open(f'{MODEL_DIR}/model_metadata.pkl', 'wb') as f:
        pickle.dump(model_meta, f)
    
    print(f"\n✅ 모든 파일 저장 완료!")

if __name__ == "__main__":
    train_model()

In [ ]:
import pandas as pd
import numpy as np
import os
import pickle
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

# 파일 경로
INPUT_FILE = './data/kaggle-dataset/vct_combined_all_years.csv'
OUTPUT_DIR = './data/seq2seq_data'
SEQUENCE_LENGTH = 12
FUTURE_STEPS = 12

def parse_k_value(x):
    """'0.3k' → 300, '4k' → 4000 변환"""
    if isinstance(x, str):
        x = x.strip()
        if x.endswith('k'):
            try:
                return float(x[:-1]) * 1000.0
            except ValueError:
                return np.nan
        try:
            return float(x.replace(',', ''))
        except ValueError:
            return np.nan
    return float(x) if pd.notna(x) else np.nan

def extract_features_robust():
    """
    Seq2Seq 모델 특성 추출 (NaN 처리 강화)
    
    Step 2: 강화된 결측치 처리
    """
    
    print("="*80)
    print("Step 2: Seq2Seq 모델 특성 추출 (강화 버전)")
    print("="*80)
    
    # 1. 데이터 로드
    print("\n[1/7] 통합 데이터 로드 중...")
    try:
        df = pd.read_csv(INPUT_FILE)
        print(f"✅ 로드 완료: {len(df):,}개 행, {df['Game ID'].nunique()}개 게임")
    except FileNotFoundError:
        print(f"❌ 파일 없음: {INPUT_FILE}")
        return False
    
    # 2. 기본 전처리 (NaN 처리 강화)
    print("\n[2/7] 기본 전처리 중...")
    
    # 숫자형 컬럼 문자열 처리
    print(f"   숫자형 변환:")
    for col in ['Loadout Value', 'Remaining Credits', 'My_Kills', 'Opp_Kills']:
        if col in df.columns:
            df[col] = df[col].apply(parse_k_value)
            null_count = df[col].isnull().sum()
            print(f"     {col}: {null_count:,}개 NaN")
    
    # Outcome 인코딩
    df['Outcome_Encoded'] = (df['Outcome'] == 'Win').astype(int)
    
    # Buy Type 인코딩
    buy_type_mapping = {'Eco': 0, 'Semi': 1, 'Force': 2, 'Full': 3}
    if 'Type' in df.columns:
        df['Buy Type Encoded'] = df['Type'].map(buy_type_mapping)
        df['Buy Type Encoded'] = df['Buy Type Encoded'].fillna(0)
    else:
        df['Buy Type Encoded'] = 0
    
    # ✅ 중요: 모든 NaN을 0으로 채우기 (앞에서 했다면 더블 체크)
    print(f"\n   결측치 채우기:")
    null_before = df[['Loadout Value', 'Remaining Credits', 'My_Kills', 'Opp_Kills', 'Cumulative Score']].isnull().sum().sum()
    
    # 게임별로 전진 채우기 (forward fill)
    for col in ['Loadout Value', 'Remaining Credits', 'Cumulative Score']:
        if col in df.columns:
            df[col] = df.groupby('Game ID')[col].transform(lambda x: x.fillna(method='ffill').fillna(0))
    
    # 킬은 0으로 채우기 (킬이 없으면 0)
    for col in ['My_Kills', 'Opp_Kills']:
        if col in df.columns:
            df[col] = df[col].fillna(0)
    
    null_after = df[['Loadout Value', 'Remaining Credits', 'My_Kills', 'Opp_Kills', 'Cumulative Score']].isnull().sum().sum()
    print(f"     전: {null_before:,}개 NaN → 후: {null_after:,}개 NaN")
    
    # 3. 게임별 시퀀스 생성
    print("\n[3/7] 게임별 시퀀스 생성 중...")
    
    sequences = []
    invalid_count = 0
    
    for game_id, game_data in df.groupby('Game ID'):
        game_data = game_data.sort_values('Round Number').reset_index(drop=True)
        
        for team in game_data['Team'].unique():
            team_seq = game_data[game_data['Team'] == team].reset_index(drop=True)
            
            if len(team_seq) < SEQUENCE_LENGTH + FUTURE_STEPS:
                invalid_count += 1
                continue
            
            opp_team = game_data[game_data['Team'] != team].reset_index(drop=True)
            
            for start_idx in range(0, len(team_seq) - SEQUENCE_LENGTH - FUTURE_STEPS + 1):
                input_seq_indices = range(start_idx, start_idx + SEQUENCE_LENGTH)
                output_seq_indices = range(start_idx + SEQUENCE_LENGTH, 
                                          start_idx + SEQUENCE_LENGTH + FUTURE_STEPS)
                
                input_seq = team_seq.iloc[list(input_seq_indices)].reset_index(drop=True)
                output_seq = team_seq.iloc[list(output_seq_indices)].reset_index(drop=True)
                opp_input_seq = opp_team.iloc[list(input_seq_indices)].reset_index(drop=True)
                opp_output_seq = opp_team.iloc[list(output_seq_indices)].reset_index(drop=True)
                
                sequences.append({
                    'game_id': game_id,
                    'team': team,
                    'input_seq': input_seq,
                    'output_seq': output_seq,
                    'opp_input_seq': opp_input_seq,
                    'opp_output_seq': opp_output_seq
                })
    
    print(f"✅ 시퀀스 생성: {len(sequences):,}개, 불완전: {invalid_count}개")
    
    # 4. 특성 벡터 변환 (NaN 체크)
    print("\n[4/7] 특성 벡터 변환 중...")
    
    X_sequences = []
    Y_sequences = []
    skip_count = 0
    
    for seq_data in sequences:
        input_seq = seq_data['input_seq']
        output_seq = seq_data['output_seq']
        opp_input_seq = seq_data['opp_input_seq']
        opp_output_seq = seq_data['opp_output_seq']
        
        X = []
        for idx in range(len(input_seq)):
            features = [
                idx,
                input_seq['Cumulative Score'].iloc[idx],
                opp_input_seq['Cumulative Score'].iloc[idx],
                input_seq['Remaining Credits'].iloc[idx] if 'Remaining Credits' in input_seq.columns else 0,
                input_seq['Loadout Value'].iloc[idx] if 'Loadout Value' in input_seq.columns else 0,
                input_seq['Buy Type Encoded'].iloc[idx],
                input_seq['My_Kills'].iloc[idx],
                input_seq['Opp_Kills'].iloc[idx],
            ]
            
            # ✅ NaN 체크
            if any(pd.isna(f) for f in features):
                skip_count += 1
                break
            
            X.append(features)
        
        if len(X) < SEQUENCE_LENGTH:
            continue
        
        Y = []
        for idx in range(len(output_seq)):
            features = [
                SEQUENCE_LENGTH + idx,
                output_seq['Cumulative Score'].iloc[idx],
                opp_output_seq['Cumulative Score'].iloc[idx],
                output_seq['Remaining Credits'].iloc[idx] if 'Remaining Credits' in output_seq.columns else 0,
                output_seq['Loadout Value'].iloc[idx] if 'Loadout Value' in output_seq.columns else 0,
                output_seq['Buy Type Encoded'].iloc[idx],
                output_seq['My_Kills'].iloc[idx],
                output_seq['Opp_Kills'].iloc[idx],
                output_seq['Outcome_Encoded'].iloc[idx],
            ]
            
            # ✅ NaN 체크
            if any(pd.isna(f) for f in features):
                skip_count += 1
                break
            
            Y.append(features)
        
        if len(Y) < FUTURE_STEPS:
            continue
        
        X_sequences.append(np.array(X, dtype=np.float32))
        Y_sequences.append(np.array(Y, dtype=np.float32))
    
    X_sequences = np.array(X_sequences)
    Y_sequences = np.array(Y_sequences)
    
    print(f"✅ 벡터 변환 완료: {len(X_sequences):,}개")
    print(f"   X 형태: {X_sequences.shape}")
    print(f"   Y 형태: {Y_sequences.shape}")
    print(f"   NaN 건너뜬 샘플: {skip_count}")
    
    # 5. 최종 NaN 체크
    print("\n[5/7] 최종 데이터 품질 검증...")
    print(f"   X NaN: {np.isnan(X_sequences).sum()}")
    print(f"   Y NaN: {np.isnan(Y_sequences).sum()}")
    print(f"   X Inf: {np.isinf(X_sequences).sum()}")
    print(f"   Y Inf: {np.isinf(Y_sequences).sum()}")
    print(f"   X 범위: [{np.nanmin(X_sequences):.4f}, {np.nanmax(X_sequences):.4f}]")
    print(f"   Y 범위: [{np.nanmin(Y_sequences):.4f}, {np.nanmax(Y_sequences):.4f}]")
    
    if np.isnan(X_sequences).sum() > 0 or np.isnan(Y_sequences).sum() > 0:
        print(f"❌ 아직도 NaN이 있습니다!")
        return False
    
    print(f"✅ 데이터 품질 OK")
    
    # 6. 데이터 정규화 (NaN이 없으므로 안전)
    print("\n[6/7] 데이터 정규화 중...")
    
    X_flat = X_sequences.reshape(-1, X_sequences.shape[-1])
    scaler_X = StandardScaler()
    X_flat_scaled = scaler_X.fit_transform(X_flat)
    X_sequences_scaled = X_flat_scaled.reshape(X_sequences.shape)
    
    Y_flat = Y_sequences.reshape(-1, Y_sequences.shape[-1])
    scaler_Y = StandardScaler()
    Y_flat_scaled = scaler_Y.fit_transform(Y_flat[:, :-1])
    Y_flat_scaled_with_outcome = np.hstack([Y_flat_scaled, Y_flat[:, -1:]])
    Y_sequences_scaled = Y_flat_scaled_with_outcome.reshape(Y_sequences.shape)
    
    print(f"✅ 정규화 완료")
    print(f"   X: mean={X_sequences_scaled.mean():.6f}, std={X_sequences_scaled.std():.6f}")
    print(f"   Y: mean={Y_sequences_scaled[:,:,:-1].mean():.6f}, std={Y_sequences_scaled[:,:,:-1].std():.6f}")
    
    # 7. 저장
    print("\n[7/7] 데이터 저장 중...")
    
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    
    np.save(f"{OUTPUT_DIR}/X_sequences.npy", X_sequences_scaled)
    np.save(f"{OUTPUT_DIR}/Y_sequences.npy", Y_sequences_scaled)
    
    with open(f"{OUTPUT_DIR}/scaler_X.pkl", 'wb') as f:
        pickle.dump(scaler_X, f)
    with open(f"{OUTPUT_DIR}/scaler_Y.pkl", 'wb') as f:
        pickle.dump(scaler_Y, f)
    
    metadata = {
        'total_sequences': len(sequences),
        'sequence_length': SEQUENCE_LENGTH,
        'future_steps': FUTURE_STEPS,
        'feature_names': [
            'Round Index', 'My Score', 'Opp Score', 'My Bank', 
            'Loadout Value', 'Buy Type', 'My Kills', 'Opp Kills'
        ],
        'target_names': [
            'Round Index', 'My Score', 'Opp Score', 'My Bank', 
            'Loadout Value', 'Buy Type', 'My Kills', 'Opp Kills', 'Round Outcome'
        ],
        'X_shape': X_sequences_scaled.shape,
        'Y_shape': Y_sequences_scaled.shape,
    }
    
    with open(f"{OUTPUT_DIR}/metadata.pkl", 'wb') as f:
        pickle.dump(metadata, f)
    
    print(f"✅ 저장 완료: {OUTPUT_DIR}/")
    
    # 최종 요약
    print(f"\n{'='*80}")
    print("✅ Step 2 완료! (NaN 처리 강화)")
    print(f"{'='*80}")
    print(f"\n데이터 통계:")
    print(f"   총 시퀀스: {len(X_sequences):,}개")
    print(f"   X 형태: {X_sequences_scaled.shape}")
    print(f"   Y 형태: {Y_sequences_scaled.shape}")
    print(f"\n✅ 이제 Step 3으로 안전하게 진행할 수 있습니다!")
    
    return True

if __name__ == "__main__":
    extract_features_robust()

In [ ]:
"""
Valorant Seq2Seq 라운드 예측 - 최종 통합 코드 (3가지 모델 비교) - 최종 수정 버전
====================================================================

Step 1: 데이터 준비
Step 2: Seq2Seq 모델 구축 및 학습
Step 3: Encoder-only 모델 구축 및 학습
Step 4: Autoregressive 모델 구축 및 평가
Step 5: 세 모델 비교 및 ROC Curve 분석
"""

import numpy as np
import pandas as pd
import os
import pickle
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    mean_squared_error, r2_score, accuracy_score, f1_score,
    roc_curve, auc, roc_auc_score, confusion_matrix, classification_report
)
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model
from tensorflow.keras.callbacks import Callback, EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.optimizers import Adam
import warnings
warnings.filterwarnings('ignore')

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 설정
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

DATA_DIR = './data/seq2seq_data'
MODEL_DIR = './models/seq2seq_comparison'
BATCH_SIZE = 16
EPOCHS = 100
SEED = 42

# ✅ 상수 정의
SEQUENCE_LENGTH = 12
FUTURE_STEPS = 12
N_INPUT_FEATURES = 8     # X: 8개 특성
N_OUTPUT_FEATURES = 9    # Y: 8개 특성 + 1개 타겟
N_DECODER_INPUT = 8      # Decoder input: 8개 특성 (타겟 제외)

np.random.seed(SEED)
tf.random.set_seed(SEED)

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# Callback: 상세 지표 계산
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

class MetricsCallback(Callback):
    """매 에포크마다 상세 지표 계산"""
    
    def __init__(self, X_val, Y_val, model_type='encoder_only'):
        super().__init__()
        self.X_val = X_val
        self.Y_val = Y_val
        self.model_type = model_type
        self.history = {
            'val_r2': [],
            'val_mae': [],
            'val_mse': [],
            'val_rmse': [],
            'val_outcome_acc': [],
            'val_outcome_f1': [],
            'val_auc': [],
        }
    
    def on_epoch_end(self, epoch, logs=None):
        """에포크 끝날 때마다 호출"""
        
        if self.model_type == 'encoder_only':
            # Encoder-only: 디코더 입력 = 0
            val_pred = self.model.predict(self.X_val, verbose=0)
        elif self.model_type == 'seq2seq':
            # Seq2Seq: 정답 봄 (검증용 Teacher Forcing)
            decoder_input = self.Y_val[:, :, :-1]  # 8개 특성
            val_pred = self.model.predict([self.X_val, decoder_input], verbose=0)
        else:
            # Autoregressive: 한 라운드씩 생성
            val_pred = np.zeros_like(self.Y_val)
            for i, X_sample in enumerate(self.X_val):
                val_pred[i] = autoregressive_predict_single(
                    self.model, X_sample, self.Y_val.shape[1], n_features=N_DECODER_INPUT
                )
        
        # 특성 (8개, 타겟 제외)
        val_pred_features = val_pred[:, :, :-1]
        val_targets_features = self.Y_val[:, :, :-1]
        
        # 타겟 (라운드 승패)
        val_pred_outcome_prob = val_pred[:, :, -1].flatten()
        val_pred_outcome = (val_pred_outcome_prob > 0.5).astype(int)
        val_targets_outcome = self.Y_val[:, :, -1].astype(int).flatten()
        
        # R² Score
        try:
            r2_scores = []
            for feat_idx in range(N_DECODER_INPUT):
                r2 = r2_score(
                    val_targets_features[:, :, feat_idx].flatten(),
                    val_pred_features[:, :, feat_idx].flatten()
                )
                r2_scores.append(r2)
            val_r2 = np.mean(r2_scores)
        except:
            val_r2 = -1.0
        
        # MAE, MSE, RMSE
        val_mae = np.mean(np.abs(val_targets_features - val_pred_features))
        val_mse = np.mean((val_targets_features - val_pred_features) ** 2)
        val_rmse = np.sqrt(val_mse)
        
        # Outcome Accuracy & F1
        val_outcome_acc = accuracy_score(val_targets_outcome, val_pred_outcome)
        val_outcome_f1 = f1_score(val_targets_outcome, val_pred_outcome, zero_division=0)
        
        # AUC
        try:
            val_auc = roc_auc_score(val_targets_outcome, val_pred_outcome_prob)
        except:
            val_auc = 0.5
        
        # History 저장
        self.history['val_r2'].append(val_r2)
        self.history['val_mae'].append(val_mae)
        self.history['val_mse'].append(val_mse)
        self.history['val_rmse'].append(val_rmse)
        self.history['val_outcome_acc'].append(val_outcome_acc)
        self.history['val_outcome_f1'].append(val_outcome_f1)
        self.history['val_auc'].append(val_auc)
        
        # 매 20 에포크마다 출력
        if (epoch + 1) % 20 == 0:
            print(f"\n[Epoch {epoch+1}] {self.model_type.upper()} Validation:")
            print(f"  R²: {val_r2:.4f}, MAE: {val_mae:.6f}, AUC: {val_auc:.4f}")
            print(f"  Outcome Acc: {val_outcome_acc:.4f}, F1: {val_outcome_f1:.4f}")

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# Autoregressive 추론 함수
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def autoregressive_predict_single(model, X_sample, future_steps, n_features=8):
    """
    단일 샘플에 대한 Autoregressive 예측
    
    한 라운드씩 예측하며 다음 입력으로 사용
    """
    
    decoder_sequence = np.zeros((1, future_steps, n_features), dtype=np.float32)
    predictions = []
    
    for step in range(future_steps):
        # 현재까지의 디코더 입력으로 예측
        pred = model.predict(
            [np.expand_dims(X_sample, 0), decoder_sequence],
            verbose=0
        )  # (1, 12, 9)
        
        # step번째 라운드 예측 (8개 특성 + 1개 타겟)
        step_pred = pred[0, step, :]  # (9,)
        predictions.append(step_pred)
        
        # 다음 스텝을 위해 현재 예측을 디코더 입력으로 사용
        if step < future_steps - 1:
            decoder_sequence[0, step + 1, :] = step_pred[:n_features]
    
    return np.array(predictions)  # (12, 9)

def autoregressive_predict_batch(model, X_batch, future_steps, n_features=8):
    """배치 단위 Autoregressive 예측"""
    
    batch_size = X_batch.shape[0]
    predictions = np.zeros((batch_size, future_steps, n_features + 1), dtype=np.float32)
    
    print(f"  Autoregressive 추론 중... ({batch_size}개)", end='', flush=True)
    for i, X_sample in enumerate(X_batch):
        predictions[i] = autoregressive_predict_single(model, X_sample, future_steps, n_features)
        
        if (i + 1) % max(1, batch_size // 10) == 0:
            print(".", end='', flush=True)
    print()
    
    return predictions

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 모델 1: Seq2Seq (Teacher Forcing)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def build_seq2seq_model():
    """
    Seq2Seq 모델 (Teacher Forcing 방식)
    
    Encoder: LSTM (64)
    Decoder: LSTM (64) + TimeDistributed Dense
    
    ✅ 상수를 사용하여 명확하게 정의
    """
    
    print("\n✅ [Seq2Seq] 모델 구축 중...")
    
    # Encoder
    encoder_inputs = keras.Input(shape=(SEQUENCE_LENGTH, N_INPUT_FEATURES), name='encoder_input')
    encoder_lstm = layers.LSTM(
        64,
        return_state=True,
        recurrent_dropout=0.2,
        name='encoder_lstm'
    )
    encoder_outputs, state_h, state_c = encoder_lstm(encoder_inputs)
    
    # Decoder (✅ 명확하게 N_DECODER_INPUT=8)
    decoder_inputs = keras.Input(shape=(FUTURE_STEPS, N_DECODER_INPUT), name='decoder_input')
    decoder_lstm = layers.LSTM(
        64,
        return_sequences=True,
        recurrent_dropout=0.2,
        name='decoder_lstm'
    )
    decoder_outputs = decoder_lstm(decoder_inputs, initial_state=[state_h, state_c])
    
    # Layer Normalization
    decoder_ln = layers.LayerNormalization()(decoder_outputs)
    
    # TimeDistributed Dense
    decoder_td = layers.TimeDistributed(
        layers.Dense(32, activation='relu')
    )(decoder_ln)
    
    decoder_dropout = layers.Dropout(0.3)(decoder_td)
    
    # 출력 (9개: 8개 특성 + 1개 타겟)
    outputs = layers.TimeDistributed(
        layers.Dense(N_OUTPUT_FEATURES)
    )(decoder_dropout)
    
    model = Model([encoder_inputs, decoder_inputs], outputs, name='Seq2Seq_Model')
    
    return model

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 모델 2: Encoder-only (RepeatVector)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def build_encoder_only_model():
    """
    Encoder-only 모델 (한 번에 생성)
    
    Encoder: Bidirectional LSTM (64)
    Decoder: RepeatVector + LSTM (128) + TimeDistributed Dense
    """
    
    print("\n✅ [Encoder-only] 모델 구축 중...")
    
    # Encoder
    encoder_inputs = keras.Input(shape=(SEQUENCE_LENGTH, N_INPUT_FEATURES), name='encoder_input')
    encoder_lstm = layers.Bidirectional(
        layers.LSTM(64, return_state=True, recurrent_dropout=0.2)
    )
    encoder_outputs, forward_h, forward_c, backward_h, backward_c = encoder_lstm(encoder_inputs)
    
    # Context Vector (128*2 = 256)
    state_h = layers.Concatenate()([forward_h, backward_h])
    state_c = layers.Concatenate()([forward_c, backward_c])
    context = layers.Concatenate()([state_h, state_c])  # (256,)
    
    # Decoder: RepeatVector
    decoder_repeated = layers.RepeatVector(FUTURE_STEPS)(context)  # (12, 256)
    
    decoder_lstm = layers.LSTM(
        128,
        return_sequences=True,
        recurrent_dropout=0.2
    )(decoder_repeated)
    
    # Layer Normalization
    decoder_ln = layers.LayerNormalization()(decoder_lstm)
    
    # TimeDistributed Dense
    decoder_td = layers.TimeDistributed(
        layers.Dense(64, activation='relu')
    )(decoder_ln)
    
    decoder_dropout = layers.Dropout(0.3)(decoder_td)
    
    # 출력 (9개: 8개 특성 + 1개 타겟)
    outputs = layers.TimeDistributed(
        layers.Dense(N_OUTPUT_FEATURES)
    )(decoder_dropout)
    
    model = Model(encoder_inputs, outputs, name='EncoderOnly_Model')
    
    return model

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 모델 3: Autoregressive (Seq2Seq 기반, 순차 생성)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def build_autoregressive_model():
    """
    Autoregressive 모델 (Seq2Seq와 동일한 구조)
    
    차이점: 추론 시 한 라운드씩 생성하며 다음 입력으로 사용
    """
    
    print("\n✅ [Autoregressive] 모델 구축 중...")
    
    # Seq2Seq와 동일한 구조 (학습은 같음, 추론만 다름)
    model = build_seq2seq_model()
    
    return model

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 데이터 로드 및 분할
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def load_and_prepare_data():
    """데이터 로드 및 분할"""
    
    print("\n"+"="*80)
    print("데이터 로드 및 준비")
    print("="*80)
    
    X = np.load(f'{DATA_DIR}/X_sequences.npy')
    Y = np.load(f'{DATA_DIR}/Y_sequences.npy')
    
    with open(f'{DATA_DIR}/metadata.pkl', 'rb') as f:
        metadata = pickle.load(f)
    
    print(f"✅ 데이터 로드: X {X.shape}, Y {Y.shape}")
    
    # Train/Val/Test 분할
    indices = np.arange(len(X))
    train_idx, test_idx = train_test_split(indices, test_size=0.2, random_state=SEED)
    train_idx, val_idx = train_test_split(train_idx, test_size=0.2/(1-0.2), random_state=SEED)
    
    X_train, Y_train = X[train_idx], Y[train_idx]
    X_val, Y_val = X[val_idx], Y[val_idx]
    X_test, Y_test = X[test_idx], Y[test_idx]
    
    print(f"\n데이터 분할:")
    print(f"  Train: {len(X_train):,}")
    print(f"  Val:   {len(X_val):,}")
    print(f"  Test:  {len(X_test):,}")
    
    return (X_train, Y_train), (X_val, Y_val), (X_test, Y_test), metadata

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 모델 학습 함수
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def train_model(model, model_type, train_data, val_data):
    """모델 학습"""
    
    print(f"\n{'='*80}")
    print(f"모델 학습: {model_type.upper()}")
    print(f"{'='*80}")
    
    X_train, Y_train = train_data
    X_val, Y_val = val_data
    
    # 컴파일
    print(f"\n[1/3] 모델 컴파일 중...")
    optimizer = Adam(learning_rate=0.0001, clipnorm=1.0)
    model.compile(optimizer=optimizer, loss='mse', metrics=['mae'])
    print(f"✅ 컴파일 완료")
    
    # Callback
    print(f"\n[2/3] Callback 설정 중...")
    metrics_callback = MetricsCallback(X_val, Y_val, model_type=model_type)
    
    callbacks = [
        EarlyStopping(monitor='val_loss', patience=20, restore_best_weights=True, verbose=1),
        ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=10, min_lr=1e-7, verbose=1),
        ModelCheckpoint(f'{MODEL_DIR}/best_{model_type}.h5', monitor='val_loss', save_best_only=True, verbose=0),
        metrics_callback
    ]
    print(f"✅ Callback 설정 완료")
    
    # 학습
    print(f"\n[3/3] 모델 학습 중...")
    
    if model_type in ['seq2seq', 'autoregressive']:
        # Seq2Seq & Autoregressive: 디코더 입력 필요 (학습은 동일)
        # ✅ Y의 처음 8개 특성만 사용 (타겟 제외)
        decoder_inputs_train = Y_train[:, :, :-1]  # (N, 12, 8) ✅
        decoder_inputs_val = Y_val[:, :, :-1]      # (N, 12, 8) ✅
        
        history = model.fit(
            [X_train, decoder_inputs_train],
            Y_train,
            batch_size=BATCH_SIZE,
            epochs=EPOCHS,
            validation_data=([X_val, decoder_inputs_val], Y_val),
            callbacks=callbacks,
            verbose=1
        )
    else:
        # Encoder-only: 인코더만
        history = model.fit(
            X_train,
            Y_train,
            batch_size=BATCH_SIZE,
            epochs=EPOCHS,
            validation_data=(X_val, Y_val),
            callbacks=callbacks,
            verbose=1
        )
    
    print(f"\n✅ 학습 완료")
    
    return history, metrics_callback

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 모델 평가 함수
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def evaluate_model(model, model_type, test_data):
    """모델 평가"""
    
    print(f"\n{'='*80}")
    print(f"모델 평가: {model_type.upper()}")
    print(f"{'='*80}")
    
    X_test, Y_test = test_data
    
    # 예측
    print(f"\n[1/2] 예측 중...")
    
    if model_type == 'seq2seq':
        # ✅ decoder_input은 8개 특성
        decoder_input = Y_test[:, :, :-1]  # (N, 12, 8) ✅
        predictions = model.predict([X_test, decoder_input], batch_size=32, verbose=0)
    elif model_type == 'encoder_only':
        predictions = model.predict(X_test, batch_size=32, verbose=0)
    else:  # autoregressive
        predictions = autoregressive_predict_batch(model, X_test, FUTURE_STEPS, n_features=N_DECODER_INPUT)
    
    print(f"✅ 예측 완료")
    
    # 특성 평가 (8개)
    print(f"\n[2/2] 평가 중...")
    
    pred_features = predictions[:, :, :-1]
    target_features = Y_test[:, :, :-1]
    
    # 타겟 평가 (라운드 승패)
    pred_outcome_prob = predictions[:, :, -1].flatten()
    pred_outcome = (pred_outcome_prob > 0.5).astype(int)
    target_outcome = Y_test[:, :, -1].astype(int).flatten()
    
    # 지표 계산
    r2_scores = []
    for feat_idx in range(N_DECODER_INPUT):
        r2 = r2_score(
            target_features[:, :, feat_idx].flatten(),
            pred_features[:, :, feat_idx].flatten()
        )
        r2_scores.append(r2)
    
    r2 = np.mean(r2_scores)
    mae = np.mean(np.abs(target_features - pred_features))
    mse = np.mean((target_features - pred_features) ** 2)
    rmse = np.sqrt(mse)
    
    # Outcome 지표
    outcome_acc = accuracy_score(target_outcome, pred_outcome)
    outcome_f1 = f1_score(target_outcome, pred_outcome, zero_division=0)
    outcome_auc = roc_auc_score(target_outcome, pred_outcome_prob)
    
    # ROC Curve 데이터
    fpr, tpr, thresholds = roc_curve(target_outcome, pred_outcome_prob)
    
    print(f"✅ 평가 완료")
    print(f"\n📊 성능 지표 (Test):")
    print(f"  R² Score: {r2:.4f}")
    print(f"  MAE: {mae:.6f}")
    print(f"  RMSE: {rmse:.6f}")
    print(f"  Outcome Accuracy: {outcome_acc:.4f} ({outcome_acc*100:.2f}%)")
    print(f"  Outcome F1-Score: {outcome_f1:.4f}")
    print(f"  Outcome AUC-ROC: {outcome_auc:.4f}")
    
    # Confusion Matrix
    cm = confusion_matrix(target_outcome, pred_outcome)
    print(f"\n📈 Confusion Matrix:")
    print(f"  True Negatives: {cm[0, 0]}, False Positives: {cm[0, 1]}")
    print(f"  False Negatives: {cm[1, 0]}, True Positives: {cm[1, 1]}")
    
    return {
        'r2': r2,
        'mae': mae,
        'mse': mse,
        'rmse': rmse,
        'outcome_accuracy': outcome_acc,
        'outcome_f1': outcome_f1,
        'outcome_auc': outcome_auc,
        'fpr': fpr,
        'tpr': tpr,
        'threshold': thresholds,
        'pred_outcome_prob': pred_outcome_prob,
        'target_outcome': target_outcome,
        'confusion_matrix': cm,
    }

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 시각화 함수
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def plot_comparison(results_dict):
    """세 모델 비교 시각화"""
    
    print(f"\n{'='*80}")
    print("시각화: 세 모델 비교")
    print(f"{'='*80}")
    
    os.makedirs(f'{MODEL_DIR}/plots', exist_ok=True)
    
    seq2seq_results = results_dict['seq2seq']
    encoder_results = results_dict['encoder_only']
    autoregressive_results = results_dict['autoregressive']
    
    # 1. ROC Curve 비교 (2x2 그리드)
    fig, axes = plt.subplots(2, 2, figsize=(15, 12))
    fig.suptitle('Seq2Seq vs Encoder-only vs Autoregressive 모델 비교', fontsize=16, fontweight='bold')
    
    # ROC Curve
    ax = axes[0, 0]
    ax.plot([0, 1], [0, 1], 'k--', label='Random Classifier', linewidth=2)
    ax.plot(seq2seq_results['fpr'], seq2seq_results['tpr'], 
            label=f"Seq2Seq (AUC = {seq2seq_results['outcome_auc']:.4f})",
            linewidth=2.5, marker='o', markersize=4)
    ax.plot(encoder_results['fpr'], encoder_results['tpr'],
            label=f"Encoder-only (AUC = {encoder_results['outcome_auc']:.4f})",
            linewidth=2.5, marker='s', markersize=4)
    ax.plot(autoregressive_results['fpr'], autoregressive_results['tpr'],
            label=f"Autoregressive (AUC = {autoregressive_results['outcome_auc']:.4f})",
            linewidth=2.5, marker='^', markersize=4)
    ax.set_xlabel('False Positive Rate', fontsize=11)
    ax.set_ylabel('True Positive Rate', fontsize=11)
    ax.set_title('ROC Curve 비교', fontsize=12, fontweight='bold')
    ax.legend(loc='lower right', fontsize=10)
    ax.grid(True, alpha=0.3)
    
    # 성능 지표 비교 (막대 그래프)
    ax = axes[0, 1]
    metrics = ['R²', 'Accuracy', 'F1-Score', 'AUC']
    seq2seq_vals = [
        seq2seq_results['r2'],
        seq2seq_results['outcome_accuracy'],
        seq2seq_results['outcome_f1'],
        seq2seq_results['outcome_auc']
    ]
    encoder_vals = [
        encoder_results['r2'],
        encoder_results['outcome_accuracy'],
        encoder_results['outcome_f1'],
        encoder_results['outcome_auc']
    ]
    autoregressive_vals = [
        autoregressive_results['r2'],
        autoregressive_results['outcome_accuracy'],
        autoregressive_results['outcome_f1'],
        autoregressive_results['outcome_auc']
    ]
    
    x = np.arange(len(metrics))
    width = 0.25
    ax.bar(x - width, seq2seq_vals, width, label='Seq2Seq', alpha=0.8)
    ax.bar(x, encoder_vals, width, label='Encoder-only', alpha=0.8)
    ax.bar(x + width, autoregressive_vals, width, label='Autoregressive', alpha=0.8)
    ax.set_ylabel('Score', fontsize=11)
    ax.set_title('성능 지표 비교', fontsize=12, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(metrics)
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3, axis='y')
    ax.set_ylim([0, 1])
    
    # Confusion Matrix 비교
    ax = axes[1, 0]
    cm_combined = np.array([
        seq2seq_results['confusion_matrix'].flatten(),
        encoder_results['confusion_matrix'].flatten(),
        autoregressive_results['confusion_matrix'].flatten()
    ])
    im = ax.imshow(cm_combined, cmap='Blues', aspect='auto')
    ax.set_xticks([0, 1, 2, 3])
    ax.set_xticklabels(['TN', 'FP', 'FN', 'TP'])
    ax.set_yticks([0, 1, 2])
    ax.set_yticklabels(['Seq2Seq', 'Encoder-only', 'Autoregressive'])
    ax.set_title('Confusion Matrix 비교', fontsize=12, fontweight='bold')
    
    # 값 표시
    for i in range(3):
        for j in range(4):
            text = ax.text(j, i, f'{cm_combined[i, j]:.0f}',
                          ha="center", va="center", color="w", fontsize=10)
    
    # MAE 비교
    ax = axes[1, 1]
    models = ['Seq2Seq', 'Encoder-only', 'Autoregressive']
    mae_vals = [seq2seq_results['mae'], encoder_results['mae'], autoregressive_results['mae']]
    colors = ['#1f77b4', '#ff7f0e', '#2ca02c']
    bars = ax.bar(models, mae_vals, color=colors, alpha=0.8, edgecolor='black', linewidth=1.5)
    ax.set_ylabel('MAE', fontsize=11)
    ax.set_title('평균 절대 오차 (MAE) 비교', fontsize=12, fontweight='bold')
    ax.grid(True, alpha=0.3, axis='y')
    
    # 값 표시
    for bar, val in zip(bars, mae_vals):
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
                f'{val:.6f}', ha='center', va='bottom', fontsize=10)
    
    plt.tight_layout()
    plt.savefig(f'{MODEL_DIR}/plots/model_comparison_3way.png', dpi=300, bbox_inches='tight')
    print(f"✅ 저장: {MODEL_DIR}/plots/model_comparison_3way.png")
    plt.close()

def plot_training_history(history_dict):
    """학습 히스토리 시각화"""
    
    seq2seq_history = history_dict['seq2seq']
    encoder_history = history_dict['encoder_only']
    autoregressive_history = history_dict['autoregressive']
    
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    fig.suptitle('모델별 학습 과정', fontsize=16, fontweight='bold')
    
    epochs = range(1, len(seq2seq_history.history['loss']) + 1)
    
    # Loss
    ax = axes[0, 0]
    ax.plot(epochs, seq2seq_history.history['loss'], 'o-', label='Seq2Seq Train', alpha=0.7)
    ax.plot(epochs, seq2seq_history.history['val_loss'], 's-', label='Seq2Seq Val', alpha=0.7)
    ax.plot(epochs, encoder_history.history['loss'], 'o--', label='Encoder-only Train', alpha=0.7)
    ax.plot(epochs, encoder_history.history['val_loss'], 's--', label='Encoder-only Val', alpha=0.7)
    ax.plot(epochs, autoregressive_history.history['loss'], 'o:', label='Autoregressive Train', alpha=0.7)
    ax.plot(epochs, autoregressive_history.history['val_loss'], 's:', label='Autoregressive Val', alpha=0.7)
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Loss')
    ax.set_title('Loss')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)
    
    # MAE
    ax = axes[0, 1]
    ax.plot(epochs, seq2seq_history.history['mae'], 'o-', label='Seq2Seq Train', alpha=0.7)
    ax.plot(epochs, seq2seq_history.history['val_mae'], 's-', label='Seq2Seq Val', alpha=0.7)
    ax.plot(epochs, encoder_history.history['mae'], 'o--', label='Encoder-only Train', alpha=0.7)
    ax.plot(epochs, encoder_history.history['val_mae'], 's--', label='Encoder-only Val', alpha=0.7)
    ax.plot(epochs, autoregressive_history.history['mae'], 'o:', label='Autoregressive Train', alpha=0.7)
    ax.plot(epochs, autoregressive_history.history['val_mae'], 's:', label='Autoregressive Val', alpha=0.7)
    ax.set_xlabel('Epoch')
    ax.set_ylabel('MAE')
    ax.set_title('MAE')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)
    
    # R²
    ax = axes[0, 2]
    ax.plot(epochs, seq2seq_history.metrics_callback.history['val_r2'], 'o-', label='Seq2Seq', alpha=0.7)
    ax.plot(epochs, encoder_history.metrics_callback.history['val_r2'], 's-', label='Encoder-only', alpha=0.7)
    ax.plot(epochs, autoregressive_history.metrics_callback.history['val_r2'], '^-', label='Autoregressive', alpha=0.7)
    ax.set_xlabel('Epoch')
    ax.set_ylabel('R²')
    ax.set_title('R² Score')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)
    
    # Outcome Accuracy
    ax = axes[1, 0]
    ax.plot(epochs, seq2seq_history.metrics_callback.history['val_outcome_acc'], 'o-', label='Seq2Seq', alpha=0.7)
    ax.plot(epochs, encoder_history.metrics_callback.history['val_outcome_acc'], 's-', label='Encoder-only', alpha=0.7)
    ax.plot(epochs, autoregressive_history.metrics_callback.history['val_outcome_acc'], '^-', label='Autoregressive', alpha=0.7)
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Accuracy')
    ax.set_title('Outcome Accuracy')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)
    
    # F1-Score
    ax = axes[1, 1]
    ax.plot(epochs, seq2seq_history.metrics_callback.history['val_outcome_f1'], 'o-', label='Seq2Seq', alpha=0.7)
    ax.plot(epochs, encoder_history.metrics_callback.history['val_outcome_f1'], 's-', label='Encoder-only', alpha=0.7)
    ax.plot(epochs, autoregressive_history.metrics_callback.history['val_outcome_f1'], '^-', label='Autoregressive', alpha=0.7)
    ax.set_xlabel('Epoch')
    ax.set_ylabel('F1-Score')
    ax.set_title('F1-Score')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)
    
    # AUC
    ax = axes[1, 2]
    ax.plot(epochs, seq2seq_history.metrics_callback.history['val_auc'], 'o-', label='Seq2Seq', alpha=0.7)
    ax.plot(epochs, encoder_history.metrics_callback.history['val_auc'], 's-', label='Encoder-only', alpha=0.7)
    ax.plot(epochs, autoregressive_history.metrics_callback.history['val_auc'], '^-', label='Autoregressive', alpha=0.7)
    ax.set_xlabel('Epoch')
    ax.set_ylabel('AUC')
    ax.set_title('AUC-ROC')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(f'{MODEL_DIR}/plots/training_history_3way.png', dpi=300, bbox_inches='tight')
    print(f"✅ 저장: {MODEL_DIR}/plots/training_history_3way.png")
    plt.close()

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 메인 함수
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def main():
    """메인 함수: 전체 파이프라인"""
    
    print("\n" + "="*80)
    print("Valorant 라운드 예측: 세 모델 비교 (Seq2Seq vs Encoder-only vs Autoregressive)")
    print("="*80)
    
    os.makedirs(MODEL_DIR, exist_ok=True)
    
    # Step 1: 데이터 로드
    train_data, val_data, test_data, metadata = load_and_prepare_data()
    
    print(f"\n✅ 데이터 구조:")
    print(f"   N_INPUT_FEATURES: {N_INPUT_FEATURES} (X: {N_INPUT_FEATURES}개 특성)")
    print(f"   N_OUTPUT_FEATURES: {N_OUTPUT_FEATURES} (Y: {N_DECODER_INPUT}개 특성 + 1개 타겟)")
    print(f"   N_DECODER_INPUT: {N_DECODER_INPUT} (Decoder input: {N_DECODER_INPUT}개 특성)")
    
    # Step 2: 모델 구축
    print(f"\n{'='*80}")
    print("모델 구축")
    print(f"{'='*80}")
    
    seq2seq_model = build_seq2seq_model()
    encoder_model = build_encoder_only_model()
    autoregressive_model = build_autoregressive_model()
    
    print(f"\n✅ 세 모델 구축 완료")
    
    # Step 3: 모델 학습
    print(f"\n{'='*80}")
    print("모델 학습")
    print(f"{'='*80}")
    
    seq2seq_history, seq2seq_metrics_callback = train_model(
        seq2seq_model, 'seq2seq', train_data, val_data
    )
    
    print(f"\n" + "-"*80 + "\n")
    
    encoder_history, encoder_metrics_callback = train_model(
        encoder_model, 'encoder_only', train_data, val_data
    )
    
    print(f"\n" + "-"*80 + "\n")
    
    autoregressive_history, autoregressive_metrics_callback = train_model(
        autoregressive_model, 'autoregressive', train_data, val_data
    )
    
    # Step 4: 모델 평가
    print(f"\n{'='*80}")
    print("모델 평가")
    print(f"{'='*80}")
    
    seq2seq_results = evaluate_model(seq2seq_model, 'seq2seq', test_data)
    encoder_results = evaluate_model(encoder_model, 'encoder_only', test_data)
    autoregressive_results = evaluate_model(autoregressive_model, 'autoregressive', test_data)
    
    # Step 5: 결과 비교
    print(f"\n{'='*80}")
    print("최종 비교 결과")
    print(f"{'='*80}")
    
    print(f"\n📊 성능 지표 비교 (Test 세트):")
    print(f"\n{'지표':<30} {'Seq2Seq':>15} {'Encoder-only':>15} {'Autoregressive':>15}")
    print(f"{'-'*75}")
    
    comparison_data = [
        ('R² Score', seq2seq_results['r2'], encoder_results['r2'], autoregressive_results['r2']),
        ('MAE', seq2seq_results['mae'], encoder_results['mae'], autoregressive_results['mae']),
        ('RMSE', seq2seq_results['rmse'], encoder_results['rmse'], autoregressive_results['rmse']),
        ('Outcome Accuracy', seq2seq_results['outcome_accuracy'], encoder_results['outcome_accuracy'], autoregressive_results['outcome_accuracy']),
        ('F1-Score', seq2seq_results['outcome_f1'], encoder_results['outcome_f1'], autoregressive_results['outcome_f1']),
        ('AUC-ROC', seq2seq_results['outcome_auc'], encoder_results['outcome_auc'], autoregressive_results['outcome_auc']),
    ]
    
    for metric_name, seq2seq_val, encoder_val, autoregressive_val in comparison_data:
        print(f"{metric_name:<30} {seq2seq_val:>15.4f} {encoder_val:>15.4f} {autoregressive_val:>15.4f}")
    
    # Step 6: 시각화
    print(f"\n{'='*80}")
    print("시각화")
    print(f"{'='*80}")
    
    results_dict = {
        'seq2seq': seq2seq_results,
        'encoder_only': encoder_results,
        'autoregressive': autoregressive_results
    }
    
    history_dict = {
        'seq2seq': seq2seq_history,
        'encoder_only': encoder_history,
        'autoregressive': autoregressive_history,
    }
    
    # 콜백 추가
    seq2seq_history.metrics_callback = seq2seq_metrics_callback
    encoder_history.metrics_callback = encoder_metrics_callback
    autoregressive_history.metrics_callback = autoregressive_metrics_callback
    
    plot_comparison(results_dict)
    plot_training_history(history_dict)
    
    # Step 7: 결과 저장
    print(f"\n{'='*80}")
    print("결과 저장")
    print(f"{'='*80}")
    
    # 모델 저장
    seq2seq_model.save(f'{MODEL_DIR}/seq2seq_model.h5')
    encoder_model.save(f'{MODEL_DIR}/encoder_only_model.h5')
    autoregressive_model.save(f'{MODEL_DIR}/autoregressive_model.h5')
    
    # 결과 저장
    final_results = {
        'seq2seq': {
            'test_metrics': seq2seq_results,
            'training_history': seq2seq_history.history,
            'metrics_callback': seq2seq_metrics_callback.history,
        },
        'encoder_only': {
            'test_metrics': encoder_results,
            'training_history': encoder_history.history,
            'metrics_callback': encoder_metrics_callback.history,
        },
        'autoregressive': {
            'test_metrics': autoregressive_results,
            'training_history': autoregressive_history.history,
            'metrics_callback': autoregressive_metrics_callback.history,
        },
        'comparison': comparison_data,
    }
    
    with open(f'{MODEL_DIR}/final_results_3way.pkl', 'wb') as f:
        pickle.dump(final_results, f)
    
    print(f"\n✅ 모델 저장: {MODEL_DIR}/")
    print(f"  - seq2seq_model.h5")
    print(f"  - encoder_only_model.h5")
    print(f"  - autoregressive_model.h5")
    print(f"  - final_results_3way.pkl")
    
    # Step 8: 최종 요약
    print(f"\n{'='*80}")
    print("✅ 전체 파이프라인 완료!")
    print(f"{'='*80}")
    
    print(f"\n📋 요약:")
    print(f"\n[1] Seq2Seq 모델")
    print(f"  - 구조: Encoder LSTM + Decoder LSTM (Teacher Forcing)")
    print(f"  - 특징: 정답 시퀀스를 보고 학습")
    print(f"  - AUC-ROC: {seq2seq_results['outcome_auc']:.4f}")
    print(f"  - Accuracy: {seq2seq_results['outcome_accuracy']*100:.2f}%")
    print(f"  - MAE: {seq2seq_results['mae']:.6f}")
    
    print(f"\n[2] Encoder-only 모델")
    print(f"  - 구조: Bidirectional LSTM + RepeatVector (한 번에 생성)")
    print(f"  - 특징: 입력만 보고 한 번에 모든 라운드 예측")
    print(f"  - AUC-ROC: {encoder_results['outcome_auc']:.4f}")
    print(f"  - Accuracy: {encoder_results['outcome_accuracy']*100:.2f}%")
    print(f"  - MAE: {encoder_results['mae']:.6f}")
    
    print(f"\n[3] Autoregressive 모델")
    print(f"  - 구조: Seq2Seq 구조 + 순차 생성 (한 라운드씩)")
    print(f"  - 특징: 이전 예측을 다음 입력으로 사용")
    print(f"  - AUC-ROC: {autoregressive_results['outcome_auc']:.4f}")
    print(f"  - Accuracy: {autoregressive_results['outcome_accuracy']*100:.2f}%")
    print(f"  - MAE: {autoregressive_results['mae']:.6f}")
    
    # 최종 승자
    results_auc = {
        'seq2seq': seq2seq_results['outcome_auc'],
        'encoder_only': encoder_results['outcome_auc'],
        'autoregressive': autoregressive_results['outcome_auc']
    }
    
    winner = max(results_auc, key=results_auc.get)
    winner_names = {'seq2seq': 'Seq2Seq', 'encoder_only': 'Encoder-only', 'autoregressive': 'Autoregressive'}
    
    print(f"\n🏆 최종 승자: {winner_names[winner]} 모델")
    print(f"   이유: 더 높은 AUC-ROC ({results_auc[winner]:.4f})")
    
    print(f"\n📊 생성된 시각화:")
    print(f"  - {MODEL_DIR}/plots/model_comparison_3way.png")
    print(f"  - {MODEL_DIR}/plots/training_history_3way.png")

if __name__ == "__main__":
    main()

In [ ]:
"""
Valorant Seq2Seq 라운드 예측 - Autoregressive 학습 (초고속 버전)
============================================================

문제: ModelCheckpoint HDF5 저장이 느림
해결: 저장 비활성화 + .keras 형식 사용 + 메모리 최적화
"""

import numpy as np
import os
import pickle
import gc
import matplotlib.pyplot as plt
from sklearn.metrics import (
    r2_score, accuracy_score, f1_score,
    roc_curve, roc_auc_score, confusion_matrix
)
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model
from tensorflow.keras.callbacks import Callback, EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam
import warnings
warnings.filterwarnings('ignore')

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 설정
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

DATA_DIR = './data/seq2seq_data'
MODEL_DIR = './models/seq2seq_comparison'
BATCH_SIZE = 16
EPOCHS = 100
SEED = 42

# 상수 정의
SEQUENCE_LENGTH = 12
FUTURE_STEPS = 12
N_INPUT_FEATURES = 8
N_OUTPUT_FEATURES = 9
N_DECODER_INPUT = 8

# ✅ GPU 메모리 최적화
physical_devices = tf.config.list_physical_devices('GPU')
if physical_devices:
    try:
        for gpu in physical_devices:
            tf.config.experimental.set_memory_growth(gpu, True)
        print(f"✅ GPU 메모리 성장 모드 활성화")
    except RuntimeError as e:
        print(e)

np.random.seed(SEED)
tf.random.set_seed(SEED)

print("\n" + "="*80)
print("✅ Autoregressive 모델 학습 (초고속 버전)")
print("="*80)

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 간단한 Callback
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

class SimpleMetricsCallback(Callback):
    """간단한 메트릭 콜백"""
    
    def __init__(self):
        super().__init__()
        self.history = {
            'val_loss': [],
            'val_mae': [],
        }
    
    def on_epoch_end(self, epoch, logs=None):
        """에포크 끝날 때마다 호출"""
        
        if logs is None:
            return
        
        # 기본 손실과 MAE만 기록
        self.history['val_loss'].append(logs.get('val_loss', 0))
        self.history['val_mae'].append(logs.get('val_mae', 0))
        
        # 매 10 에포크마다 출력
        if (epoch + 1) % 10 == 0:
            print(f"\n[Epoch {epoch+1}] Autoregressive Training:")
            print(f"  Train Loss: {logs.get('loss', 0):.6f}, Val Loss: {logs.get('val_loss', 0):.6f}")

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# Autoregressive 추론 함수 (평가용)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def autoregressive_predict_single(model, X_sample, future_steps, n_features=8):
    """단일 샘플에 대한 Autoregressive 예측"""
    
    decoder_sequence = np.zeros((1, future_steps, n_features), dtype=np.float32)
    predictions = []
    
    for step in range(future_steps):
        pred = model.predict(
            [np.expand_dims(X_sample, 0), decoder_sequence],
            verbose=0
        )
        step_pred = pred[0, step, :]
        predictions.append(step_pred)
        
        if step < future_steps - 1:
            decoder_sequence[0, step + 1, :] = step_pred[:n_features]
    
    return np.array(predictions)

def autoregressive_predict_batch(model, X_batch, future_steps, n_features=8):
    """배치 단위 Autoregressive 예측"""
    
    batch_size = X_batch.shape[0]
    predictions = np.zeros((batch_size, future_steps, n_features + 1), dtype=np.float32)
    
    print(f"  Autoregressive 추론 중... ({batch_size}개)", end='', flush=True)
    
    for i, X_sample in enumerate(X_batch):
        predictions[i] = autoregressive_predict_single(model, X_sample, future_steps, n_features)
        
        if (i + 1) % max(1, batch_size // 10) == 0:
            print(".", end='', flush=True)
    
    print()
    return predictions

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 모델: Autoregressive
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def build_autoregressive_model():
    """Autoregressive 모델 구축"""
    
    print("\n✅ [Autoregressive] 모델 구축 중...")
    
    # Encoder
    encoder_inputs = keras.Input(shape=(SEQUENCE_LENGTH, N_INPUT_FEATURES), name='encoder_input')
    encoder_lstm = layers.LSTM(
        64,
        return_state=True,
        recurrent_dropout=0.2,
        name='encoder_lstm'
    )
    encoder_outputs, state_h, state_c = encoder_lstm(encoder_inputs)
    
    # Decoder
    decoder_inputs = keras.Input(shape=(FUTURE_STEPS, N_DECODER_INPUT), name='decoder_input')
    decoder_lstm = layers.LSTM(
        64,
        return_sequences=True,
        recurrent_dropout=0.2,
        name='decoder_lstm'
    )
    decoder_outputs = decoder_lstm(decoder_inputs, initial_state=[state_h, state_c])
    
    # Layer Normalization
    decoder_ln = layers.LayerNormalization()(decoder_outputs)
    
    # TimeDistributed Dense
    decoder_td = layers.TimeDistributed(
        layers.Dense(32, activation='relu')
    )(decoder_ln)
    
    decoder_dropout = layers.Dropout(0.3)(decoder_td)
    
    # 출력
    outputs = layers.TimeDistributed(
        layers.Dense(N_OUTPUT_FEATURES)
    )(decoder_dropout)
    
    model = Model([encoder_inputs, decoder_inputs], outputs, name='Autoregressive_Model')
    
    return model

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 데이터 로드
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

print("\n"+"="*80)
print("데이터 로드")
print("="*80)

X = np.load(f'{DATA_DIR}/X_sequences.npy')
Y = np.load(f'{DATA_DIR}/Y_sequences.npy')

with open(f'{DATA_DIR}/metadata.pkl', 'rb') as f:
    metadata = pickle.load(f)

print(f"✅ 데이터 로드 완료: X {X.shape}, Y {Y.shape}")

# Train/Val/Test 분할
from sklearn.model_selection import train_test_split
indices = np.arange(len(X))
train_idx, test_idx = train_test_split(indices, test_size=0.2, random_state=SEED)
train_idx, val_idx = train_test_split(train_idx, test_size=0.2/(1-0.2), random_state=SEED)

X_train, Y_train = X[train_idx], Y[train_idx]
X_val, Y_val = X[val_idx], Y[val_idx]
X_test, Y_test = X[test_idx], Y[test_idx]

print(f"✅ 데이터 분할:")
print(f"   Train: {len(X_train):,}, Val: {len(X_val):,}, Test: {len(X_test):,}")

# 메모리 절약을 위해 원본 삭제
del X, Y, indices
gc.collect()

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# Step 1: Autoregressive 모델 학습
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

print("\n"+"="*80)
print("Step 1: Autoregressive 모델 학습 (저장 없음)")
print("="*80)

autoregressive_model = build_autoregressive_model()

print(f"\n[1/3] 모델 컴파일 중...")
optimizer = Adam(learning_rate=0.0001, clipnorm=1.0)
autoregressive_model.compile(optimizer=optimizer, loss='mse', metrics=['mae'])
print(f"✅ 컴파일 완료")

print(f"\n[2/3] Callback 설정 중...")
# ✅ ModelCheckpoint 제거 (저장 느림)
simple_metrics_callback = SimpleMetricsCallback()

callbacks = [
    EarlyStopping(monitor='val_loss', patience=20, restore_best_weights=True, verbose=0),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=10, min_lr=1e-7, verbose=0),
    simple_metrics_callback
]
print(f"✅ Callback 설정 완료")

print(f"\n[3/3] 모델 학습 중...")
print(f"    배치 크기: {BATCH_SIZE}, 에포크: {EPOCHS}")
print(f"    (주의: 저장 없이 진행하므로 빠름!)")

decoder_inputs_train = Y_train[:, :, :-1]
decoder_inputs_val = Y_val[:, :, :-1]

try:
    autoregressive_history = autoregressive_model.fit(
        [X_train, decoder_inputs_train],
        Y_train,
        batch_size=BATCH_SIZE,
        epochs=EPOCHS,
        validation_data=([X_val, decoder_inputs_val], Y_val),
        callbacks=callbacks,
        verbose=0  # ✅ verbose=0으로 로그 최소화
    )
    
    print(f"\n✅ Autoregressive 모델 학습 완료!")
    
except KeyboardInterrupt:
    print(f"\n⚠️ 사용자에 의해 중단됨")
    autoregressive_history = None
except Exception as e:
    print(f"\n❌ 에러 발생: {str(e)}")
    autoregressive_history = None

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# Step 2: Autoregressive 모델 평가
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

if autoregressive_history is not None:
    print("\n"+"="*80)
    print("Step 2: Autoregressive 모델 평가")
    print("="*80)
    
    print(f"\n[1/2] 예측 중...")
    autoregressive_predictions = autoregressive_predict_batch(
        autoregressive_model, X_test, FUTURE_STEPS, n_features=N_DECODER_INPUT
    )
    print(f"✅ 예측 완료")
    
    print(f"\n[2/2] 지표 계산 중...")
    
    # 특성 평가
    autoregressive_pred_features = autoregressive_predictions[:, :, :-1]
    autoregressive_target_features = Y_test[:, :, :-1]
    
    # 타겟 평가
    autoregressive_pred_outcome_prob = autoregressive_predictions[:, :, -1].flatten()
    autoregressive_pred_outcome = (autoregressive_pred_outcome_prob > 0.5).astype(int)
    autoregressive_target_outcome = Y_test[:, :, -1].astype(int).flatten()
    
    # 지표 계산
    autoregressive_r2_scores = []
    for feat_idx in range(N_DECODER_INPUT):
        try:
            r2 = r2_score(
                autoregressive_target_features[:, :, feat_idx].flatten(),
                autoregressive_pred_features[:, :, feat_idx].flatten()
            )
            autoregressive_r2_scores.append(r2)
        except:
            autoregressive_r2_scores.append(0)
    
    autoregressive_r2 = np.mean(autoregressive_r2_scores)
    autoregressive_mae = np.mean(np.abs(autoregressive_target_features - autoregressive_pred_features))
    autoregressive_mse = np.mean((autoregressive_target_features - autoregressive_pred_features) ** 2)
    autoregressive_rmse = np.sqrt(autoregressive_mse)
    
    autoregressive_outcome_acc = accuracy_score(autoregressive_target_outcome, autoregressive_pred_outcome)
    autoregressive_outcome_f1 = f1_score(autoregressive_target_outcome, autoregressive_pred_outcome, zero_division=0)
    autoregressive_outcome_auc = roc_auc_score(autoregressive_target_outcome, autoregressive_pred_outcome_prob)
    
    autoregressive_fpr, autoregressive_tpr, _ = roc_curve(autoregressive_target_outcome, autoregressive_pred_outcome_prob)
    autoregressive_cm = confusion_matrix(autoregressive_target_outcome, autoregressive_pred_outcome)
    
    print(f"✅ 지표 계산 완료")
    
    print(f"\n📊 Autoregressive 성능 지표 (Test):")
    print(f"  ├─ R² Score: {autoregressive_r2:.4f}")
    print(f"  ├─ MAE: {autoregressive_mae:.6f}")
    print(f"  ├─ RMSE: {autoregressive_rmse:.6f}")
    print(f"  ├─ Outcome Accuracy: {autoregressive_outcome_acc:.4f} ({autoregressive_outcome_acc*100:.2f}%)")
    print(f"  ├─ Outcome F1-Score: {autoregressive_outcome_f1:.4f}")
    print(f"  └─ Outcome AUC-ROC: {autoregressive_outcome_auc:.4f} ⭐")
    
    # ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
    # Step 3: 모델 저장 (.keras 형식 - 빠름!)
    # ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
    
    print("\n"+"="*80)
    print("Step 3: 모델 및 결과 저장")
    print("="*80)
    
    # ✅ .keras 형식으로 저장 (HDF5보다 빠름!)
    print(f"\n[1/2] 모델 저장 중...", end='', flush=True)
    autoregressive_model.save(f'{MODEL_DIR}/autoregressive_model.keras')
    print(f" ✅")
    
    # 결과 저장
    print(f"[2/2] 결과 저장 중...", end='', flush=True)
    
    autoregressive_results = {
        'r2': float(autoregressive_r2),
        'mae': float(autoregressive_mae),
        'mse': float(autoregressive_mse),
        'rmse': float(autoregressive_rmse),
        'outcome_accuracy': float(autoregressive_outcome_acc),
        'outcome_f1': float(autoregressive_outcome_f1),
        'outcome_auc': float(autoregressive_outcome_auc),
        'fpr': autoregressive_fpr.tolist(),
        'tpr': autoregressive_tpr.tolist(),
        'pred_outcome_prob': autoregressive_pred_outcome_prob.tolist(),
        'target_outcome': autoregressive_target_outcome.tolist(),
        'confusion_matrix': autoregressive_cm.tolist(),
    }
    
    final_results = {
        'autoregressive': {
            'test_metrics': autoregressive_results,
            'training_history': autoregressive_history.history if autoregressive_history else {},
            'simple_metrics': simple_metrics_callback.history,
        }
    }
    
    with open(f'{MODEL_DIR}/autoregressive_results.pkl', 'wb') as f:
        pickle.dump(final_results, f)
    
    print(f" ✅")
    
    # ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
    # Step 4: 최종 요약
    # ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
    
    print("\n"+"="*80)
    print("✅ 완료!")
    print("="*80)
    
    print(f"\n🏆 최종 결과:")
    print(f"  R² Score: {autoregressive_r2:.4f}")
    print(f"  MAE: {autoregressive_mae:.6f}")
    print(f"  RMSE: {autoregressive_rmse:.6f}")
    print(f"  Accuracy: {autoregressive_outcome_acc*100:.2f}%")
    print(f"  F1-Score: {autoregressive_outcome_f1:.4f}")
    print(f"  AUC-ROC: {autoregressive_outcome_auc:.4f} ⭐")
    
    print(f"\n📁 저장된 파일:")
    print(f"  ├─ {MODEL_DIR}/autoregressive_model.keras")
    print(f"  └─ {MODEL_DIR}/autoregressive_results.pkl")
    
    print(f"\n💾 결과 요약 (복사용):")
    print(f"\n| 항목 | 값 |")
    print(f"|------|-----|")
    print(f"| R² Score | {autoregressive_r2:.4f} |")
    print(f"| MAE | {autoregressive_mae:.6f} |")
    print(f"| RMSE | {autoregressive_rmse:.6f} |")
    print(f"| Accuracy | {autoregressive_outcome_acc*100:.2f}% |")
    print(f"| F1-Score | {autoregressive_outcome_f1:.4f} |")
    print(f"| AUC-ROC | {autoregressive_outcome_auc:.4f} |")
    
else:
    print(f"\n❌ 학습이 완료되지 않았습니다")

In [ ]:

# 모델 저장
autoregressive_model.save(f'{MODEL_DIR}/autoregressive_model.keras')
# seq2seq_model.save(f'{MODEL_DIR}/seq2seq_model.h5')
# encoder_model.save(f'{MODEL_DIR}/encoder_only_model.h5')

In [ ]:

import numpy as np
import os
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    r2_score, accuracy_score, f1_score,
    roc_auc_score, confusion_matrix, roc_curve
)
import tensorflow as tf
from tensorflow import keras
import warnings
warnings.filterwarnings('ignore')

# 설정
DATA_DIR = './data/seq2seq_data'
MODEL_DIR = './models/seq2seq_comparison'
SEED = 42

SEQUENCE_LENGTH = 12
FUTURE_STEPS = 12
N_INPUT_FEATURES = 8
N_OUTPUT_FEATURES = 9
N_DECODER_INPUT = 8

np.random.seed(SEED)
tf.random.set_seed(SEED)

print("\n" + "="*80)
print("Autoregressive 빠른 평가")
print("="*80)

# 데이터 로드
print("\n데이터 로드 중...")
X = np.load(f'{DATA_DIR}/X_sequences.npy')
Y = np.load(f'{DATA_DIR}/Y_sequences.npy')

indices = np.arange(len(X))
train_idx, test_idx = train_test_split(indices, test_size=0.2, random_state=SEED)
train_idx, val_idx = train_test_split(train_idx, test_size=0.2/(1-0.2), random_state=SEED)

X_test, Y_test = X[test_idx], Y[test_idx]
print(f"✅ 로드 완료: Test {len(X_test):,} 샘플")

# 모델 로드
print("\n모델 로드 중...")
try:
    autoregressive_model = keras.models.load_model(f'{MODEL_DIR}/autoregressive_model.keras')
    print(f"✅ 모델 로드 완료")
except:
    print(f"❌ 모델 파일 없음: {MODEL_DIR}/autoregressive_model.keras")
    exit()

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 평가 옵션 선택
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

print("\n" + "="*80)
print("평가 옵션 선택")
print("="*80)

print(f"\n평가 크기 선택:")
print(f"  1️⃣  전체 평가 (2,400개) - 5-10분 | 가장 정확")
print(f"  2️⃣  빠른 평가 (500개) - 1-2분 | 권장")
print(f"  3️⃣  초고속 평가 (200개) - 30초 | 빠름")

choice = input(f"\n선택 (1-3): ").strip()

if choice == '1':
    eval_size = len(X_test)
    print(f"\n✅ 선택: 전체 평가 ({eval_size}개)")
elif choice == '2':
    eval_size = 500
    print(f"\n✅ 선택: 빠른 평가 ({eval_size}개)")
elif choice == '3':
    eval_size = 200
    print(f"\n✅ 선택: 초고속 평가 ({eval_size}개)")
else:
    eval_size = 500
    print(f"\n⚠️ 잘못된 선택, 기본값(500)으로 진행")

# 평가 데이터 준비
X_test_eval = X_test[:eval_size]
Y_test_eval = Y_test[:eval_size]

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# Autoregressive 예측 (최적화 버전)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def autoregressive_predict_single(model, X_sample, future_steps, n_features=8):
    """단일 샘플 예측"""
    decoder_sequence = np.zeros((1, future_steps, n_features), dtype=np.float32)
    predictions = []
    
    for step in range(future_steps):
        pred = model.predict([np.expand_dims(X_sample, 0), decoder_sequence], verbose=0)
        step_pred = pred[0, step, :]
        predictions.append(step_pred)
        
        if step < future_steps - 1:
            decoder_sequence[0, step + 1, :] = step_pred[:n_features]
    
    return np.array(predictions)

print("\n" + "="*80)
print(f"예측 진행 중 ({eval_size}개 샘플)...")
print("="*80)

# ✅ 진행도 표시
import sys
predictions = np.zeros((eval_size, FUTURE_STEPS, N_OUTPUT_FEATURES), dtype=np.float32)

for i in range(eval_size):
    predictions[i] = autoregressive_predict_single(
        autoregressive_model, X_test_eval[i], FUTURE_STEPS, N_DECODER_INPUT
    )
    
    # 진행도
    progress = (i + 1) / eval_size * 100
    bar_length = 40
    filled = int(bar_length * (i + 1) / eval_size)
    bar = '█' * filled + '░' * (bar_length - filled)
    
    print(f'\r[{bar}] {progress:.1f}% ({i+1}/{eval_size})', end='', flush=True)

print(f"\n✅ 예측 완료\n")

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 평가
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

print("평가 지표 계산 중...")

pred_features = predictions[:, :, :-1]
target_features = Y_test_eval[:, :, :-1]

pred_outcome_prob = predictions[:, :, -1].flatten()
pred_outcome = (pred_outcome_prob > 0.5).astype(int)
target_outcome = Y_test_eval[:, :, -1].astype(int).flatten()

# R² Score
r2_scores = []
for feat_idx in range(N_DECODER_INPUT):
    r2 = r2_score(
        target_features[:, :, feat_idx].flatten(),
        pred_features[:, :, feat_idx].flatten()
    )
    r2_scores.append(r2)

results = {
    'r2': np.mean(r2_scores),
    'mae': np.mean(np.abs(target_features - pred_features)),
    'mse': np.mean((target_features - pred_features) ** 2),
    'rmse': np.sqrt(np.mean((target_features - pred_features) ** 2)),
    'outcome_accuracy': accuracy_score(target_outcome, pred_outcome),
    'outcome_f1': f1_score(target_outcome, pred_outcome, zero_division=0),
    'outcome_auc': roc_auc_score(target_outcome, pred_outcome_prob),
    'confusion_matrix': confusion_matrix(target_outcome, pred_outcome),
}
results['fpr'], results['tpr'], _ = roc_curve(target_outcome, pred_outcome_prob)

print("✅ 평가 완료\n")

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 결과 출력
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

print("="*80)
print("📊 Autoregressive 평가 결과")
print("="*80)

print(f"\n평가 데이터: {eval_size}개 샘플")
print(f"(전체 Test: {len(X_test):,}개 중 {eval_size/len(X_test)*100:.1f}%)")

print(f"\n📈 성능 지표:")
print(f"  R² Score: {results['r2']:.4f}")
print(f"  MAE: {results['mae']:.6f}")
print(f"  RMSE: {results['rmse']:.6f}")
print(f"  Outcome Accuracy: {results['outcome_accuracy']:.4f} ({results['outcome_accuracy']*100:.2f}%)")
print(f"  F1-Score: {results['outcome_f1']:.4f}")
print(f"  AUC-ROC: {results['outcome_auc']:.4f}")

print(f"\n📊 Confusion Matrix:")
cm = results['confusion_matrix']
print(f"  TN={cm[0,0]}, FP={cm[0,1]}")
print(f"  FN={cm[1,0]}, TP={cm[1,1]}")

# 계산
sensitivity = cm[1,1] / (cm[1,1] + cm[1,0]) if (cm[1,1] + cm[1,0]) > 0 else 0
specificity = cm[0,0] / (cm[0,0] + cm[0,1]) if (cm[0,0] + cm[0,1]) > 0 else 0
precision = cm[1,1] / (cm[1,1] + cm[0,1]) if (cm[1,1] + cm[0,1]) > 0 else 0

print(f"\n  Sensitivity (TPR): {sensitivity*100:.2f}%")
print(f"  Specificity (TNR): {specificity*100:.2f}%")
print(f"  Precision: {precision*100:.2f}%")

print(f"\n" + "="*80)
print("✅ 평가 완료!")
print("="*80)

# 결과 저장
import pickle
with open(f'{MODEL_DIR}/autoregressive_eval_results.pkl', 'wb') as f:
    pickle.dump(results, f)

print(f"\n저장: {MODEL_DIR}/autoregressive_eval_results.pkl")

In [ ]:
"""
Valorant Seq2Seq 라운드 예측 - 최종 통합 코드 (3가지 모델 비교)
=================================================================

개선된 모델 아키텍처 (깊고 견고함):
1. Seq2Seq: Multi-layer LSTM + Batch Normalization + Dropout + Dense layers
2. Encoder-only: Bidirectional Multi-layer LSTM + Batch Normalization + Dropout
3. Autoregressive: Advanced Seq2Seq with Attention-like mechanisms

특징:
- BatchNormalization: 각 LSTM 출력 후 정규화
- Dropout: 0.3~0.5 범위로 충분함
- Dense layers: 더 깊은 구조 (3-4단계)
- Residual connections: 안정적인 학습
- Layer Normalization 병행: 더 나은 수렴
"""

import numpy as np
import pandas as pd
import os
import pickle
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    mean_squared_error, r2_score, accuracy_score, f1_score,
    roc_curve, auc, roc_auc_score, confusion_matrix, classification_report
)
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model
from tensorflow.keras.callbacks import Callback, EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.optimizers import Adam
import warnings
warnings.filterwarnings('ignore')

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 전역 변수
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

seq2seq_model = None
encoder_model = None
autoregressive_model = None

seq2seq_results = None
encoder_results = None
autoregressive_results = None

X_train = None
Y_train = None
X_val = None
Y_val = None
X_test = None
Y_test = None

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 설정
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

DATA_DIR = './data/seq2seq_data'
MODEL_DIR = './models/seq2seq_comparison'
BATCH_SIZE = 16
EPOCHS = 100
SEED = 42

SEQUENCE_LENGTH = 12
FUTURE_STEPS = 12
N_INPUT_FEATURES = 8
N_OUTPUT_FEATURES = 9
N_DECODER_INPUT = 8

np.random.seed(SEED)
tf.random.set_seed(SEED)

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# Callback: 상세 지표 계산
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

class MetricsCallback(Callback):
    """매 에포크마다 상세 지표 계산"""
    
    def __init__(self, X_val, Y_val, model_type='encoder_only'):
        super().__init__()
        self.X_val = X_val
        self.Y_val = Y_val
        self.model_type = model_type
        self.history = {
            'val_r2': [],
            'val_mae': [],
            'val_outcome_acc': [],
            'val_outcome_f1': [],
            'val_auc': [],
        }
    
    def on_epoch_end(self, epoch, logs=None):
        """에포크 끝날 때마다 호출"""
        
        if self.model_type == 'encoder_only':
            val_pred = self.model.predict(self.X_val, verbose=0)
        elif self.model_type == 'seq2seq':
            decoder_input = self.Y_val[:, :, :-1]
            val_pred = self.model.predict([self.X_val, decoder_input], verbose=0)
        else:  # autoregressive
            val_pred = autoregressive_predict_batch_fast(self.model, self.X_val)
        
        val_pred_features = val_pred[:, :, :-1]
        val_targets_features = self.Y_val[:, :, :-1]
        
        val_pred_outcome_prob = val_pred[:, :, -1].flatten()
        val_pred_outcome = (val_pred_outcome_prob > 0.5).astype(int)
        val_targets_outcome = self.Y_val[:, :, -1].astype(int).flatten()
        
        try:
            r2_scores = []
            for feat_idx in range(N_DECODER_INPUT):
                r2 = r2_score(
                    val_targets_features[:, :, feat_idx].flatten(),
                    val_pred_features[:, :, feat_idx].flatten()
                )
                r2_scores.append(r2)
            val_r2 = np.mean(r2_scores)
        except:
            val_r2 = -1.0
        
        val_mae = np.mean(np.abs(val_targets_features - val_pred_features))
        val_outcome_acc = accuracy_score(val_targets_outcome, val_pred_outcome)
        val_outcome_f1 = f1_score(val_targets_outcome, val_pred_outcome, zero_division=0)
        
        try:
            val_auc = roc_auc_score(val_targets_outcome, val_pred_outcome_prob)
        except:
            val_auc = 0.5
        
        self.history['val_r2'].append(val_r2)
        self.history['val_mae'].append(val_mae)
        self.history['val_outcome_acc'].append(val_outcome_acc)
        self.history['val_outcome_f1'].append(val_outcome_f1)
        self.history['val_auc'].append(val_auc)
        
        if (epoch + 1) % 20 == 0:
            print(f"\n[Epoch {epoch+1}] {self.model_type.upper()} Validation:")
            print(f"  R²: {val_r2:.4f}, MAE: {val_mae:.6f}, AUC: {val_auc:.4f}")
            print(f"  Outcome Acc: {val_outcome_acc:.4f}, F1: {val_outcome_f1:.4f}")

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# Autoregressive 배치 처리
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def autoregressive_predict_batch_fast(model, X_batch, batch_size=32):
    """배치 단위 Autoregressive 예측"""
    
    total_samples = len(X_batch)
    predictions = np.zeros((total_samples, FUTURE_STEPS, N_OUTPUT_FEATURES), dtype=np.float32)
    
    for batch_start in range(0, total_samples, batch_size):
        batch_end = min(batch_start + batch_size, total_samples)
        X_batch_slice = X_batch[batch_start:batch_end]
        
        batch_decoder_inputs = np.zeros((len(X_batch_slice), FUTURE_STEPS, N_DECODER_INPUT), dtype=np.float32)
        batch_preds = model.predict([X_batch_slice, batch_decoder_inputs], verbose=0)
        predictions[batch_start:batch_end] = batch_preds
        
        percent = (batch_end / total_samples) * 100
        print(f"    [{batch_end}/{total_samples}] {percent:.1f}%", end='\r', flush=True)
    
    print(f"    [{total_samples}/{total_samples}] 100.0%")
    
    return predictions

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 모델 1: Seq2Seq (심화 버전)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def build_seq2seq_model():
    """
    심화된 Seq2Seq 모델
    - Encoder: 3-layer Bidirectional LSTM (64 → 128 → 64)
    - Decoder: 3-layer LSTM (128 → 256 → 128) + Dense layers
    - BatchNormalization + Dropout + LayerNormalization
    - Residual connections
    """
    
    print("\n✅ [Seq2Seq] 심화 모델 구축 중...")
    
    # ━━━━━━━ Encoder ━━━━━━━
    encoder_inputs = keras.Input(shape=(SEQUENCE_LENGTH, N_INPUT_FEATURES), name='encoder_input')
    
    # Encoder Layer 1
    encoder_layer1 = layers.Bidirectional(
        layers.LSTM(64, return_sequences=True, recurrent_dropout=0.3, dropout=0.3),
        name='encoder_bilstm1'
    )(encoder_inputs)
    encoder_bn1 = layers.BatchNormalization()(encoder_layer1)
    encoder_ln1 = layers.LayerNormalization()(encoder_bn1)
    
    # Encoder Layer 2
    encoder_layer2 = layers.Bidirectional(
        layers.LSTM(128, return_sequences=True, recurrent_dropout=0.3, dropout=0.3),
        name='encoder_bilstm2'
    )(encoder_ln1)
    encoder_bn2 = layers.BatchNormalization()(encoder_layer2)
    encoder_ln2 = layers.LayerNormalization()(encoder_bn2)
    
    # Encoder Layer 3
    encoder_layer3 = layers.Bidirectional(
        layers.LSTM(64, return_state=True, recurrent_dropout=0.3, dropout=0.3),
        name='encoder_bilstm3'
    )
    encoder_outputs, forward_h, forward_c, backward_h, backward_c = encoder_layer3(encoder_ln2)
    encoder_bn3 = layers.BatchNormalization()(encoder_outputs)
    
    # Context vector
    state_h = layers.Concatenate()([forward_h, backward_h])
    state_c = layers.Concatenate()([forward_c, backward_c])
    context = layers.Concatenate()([state_h, state_c])
    context_dense = layers.Dense(256, activation='relu')(context)
    context_bn = layers.BatchNormalization()(context_dense)
    
    # ━━━━━━━ Decoder ━━━━━━━
    decoder_inputs = keras.Input(shape=(FUTURE_STEPS, N_DECODER_INPUT), name='decoder_input')
    
    # Decoder Layer 1
    decoder_layer1 = layers.LSTM(128, return_sequences=True, recurrent_dropout=0.3, dropout=0.3, name='decoder_lstm1')
    decoder_outputs1 = decoder_layer1(decoder_inputs, initial_state=[state_h, state_c])
    decoder_bn1 = layers.BatchNormalization()(decoder_outputs1)
    decoder_ln1 = layers.LayerNormalization()(decoder_bn1)
    decoder_do1 = layers.Dropout(0.4)(decoder_ln1)
    
    # Decoder Layer 2
    decoder_layer2 = layers.LSTM(256, return_sequences=True, recurrent_dropout=0.3, dropout=0.3, name='decoder_lstm2')
    decoder_outputs2 = decoder_layer2(decoder_do1)
    decoder_bn2 = layers.BatchNormalization()(decoder_outputs2)
    decoder_ln2 = layers.LayerNormalization()(decoder_bn2)
    decoder_do2 = layers.Dropout(0.4)(decoder_ln2)
    
    # Decoder Layer 3
    decoder_layer3 = layers.LSTM(128, return_sequences=True, recurrent_dropout=0.3, dropout=0.3, name='decoder_lstm3')
    decoder_outputs3 = decoder_layer3(decoder_do2)
    decoder_bn3 = layers.BatchNormalization()(decoder_outputs3)
    decoder_ln3 = layers.LayerNormalization()(decoder_bn3)
    decoder_do3 = layers.Dropout(0.3)(decoder_ln3)
    
    # Dense layers (깊은 구조)
    dense1 = layers.TimeDistributed(layers.Dense(128, activation='relu'))(decoder_do3)
    dense1_bn = layers.TimeDistributed(layers.BatchNormalization())(dense1)
    dense1_do = layers.TimeDistributed(layers.Dropout(0.3))(dense1_bn)
    
    dense2 = layers.TimeDistributed(layers.Dense(64, activation='relu'))(dense1_do)
    dense2_bn = layers.TimeDistributed(layers.BatchNormalization())(dense2)
    dense2_do = layers.TimeDistributed(layers.Dropout(0.3))(dense2_bn)
    
    dense3 = layers.TimeDistributed(layers.Dense(32, activation='relu'))(dense2_do)
    dense3_bn = layers.TimeDistributed(layers.BatchNormalization())(dense3)
    
    # Output layer
    outputs = layers.TimeDistributed(layers.Dense(N_OUTPUT_FEATURES))(dense3_bn)
    
    model = Model([encoder_inputs, decoder_inputs], outputs, name='Seq2Seq_Advanced_Model')
    
    return model

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 모델 2: Encoder-only (심화 버전)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def build_encoder_only_model():
    """
    심화된 Encoder-only 모델
    - Encoder: 3-layer Bidirectional LSTM (64 → 128 → 64)
    - Dense feature extraction
    - Decoder: RepeatVector + 3-layer LSTM (256 → 512 → 256)
    - BatchNormalization + Dropout
    """
    
    print("\n✅ [Encoder-only] 심화 모델 구축 중...")
    
    # ━━━━━━━ Encoder ━━━━━━━
    encoder_inputs = keras.Input(shape=(SEQUENCE_LENGTH, N_INPUT_FEATURES), name='encoder_input')
    
    # Encoder Layer 1
    encoder_layer1 = layers.Bidirectional(
        layers.LSTM(64, return_sequences=True, recurrent_dropout=0.3, dropout=0.3),
        name='encoder_bilstm1'
    )(encoder_inputs)
    encoder_bn1 = layers.BatchNormalization()(encoder_layer1)
    encoder_do1 = layers.Dropout(0.3)(encoder_bn1)
    
    # Encoder Layer 2
    encoder_layer2 = layers.Bidirectional(
        layers.LSTM(128, return_sequences=True, recurrent_dropout=0.3, dropout=0.3),
        name='encoder_bilstm2'
    )(encoder_do1)
    encoder_bn2 = layers.BatchNormalization()(encoder_layer2)
    encoder_do2 = layers.Dropout(0.3)(encoder_bn2)
    
    # Encoder Layer 3
    encoder_layer3 = layers.Bidirectional(
        layers.LSTM(64, return_state=True, recurrent_dropout=0.3, dropout=0.3),
        name='encoder_bilstm3'
    )
    encoder_outputs, forward_h, forward_c, backward_h, backward_c = encoder_layer3(encoder_do2)
    encoder_bn3 = layers.BatchNormalization()(encoder_outputs)
    
    # Context vector (더 복잡한 구조)
    state_h = layers.Concatenate()([forward_h, backward_h])
    state_c = layers.Concatenate()([forward_c, backward_c])
    
    context_1 = layers.Concatenate()([state_h, state_c])
    context_dense1 = layers.Dense(512, activation='relu')(context_1)
    context_bn1 = layers.BatchNormalization()(context_dense1)
    context_do1 = layers.Dropout(0.4)(context_bn1)
    
    context_dense2 = layers.Dense(256, activation='relu')(context_do1)
    context_bn2 = layers.BatchNormalization()(context_dense2)
    context_do2 = layers.Dropout(0.4)(context_bn2)
    
    context_final = layers.Dense(256, activation='relu')(context_do2)
    
    # ━━━━━━━ Decoder ━━━━━━━
    decoder_repeated = layers.RepeatVector(FUTURE_STEPS)(context_final)
    
    # Decoder Layer 1
    decoder_layer1 = layers.LSTM(256, return_sequences=True, recurrent_dropout=0.3, dropout=0.3, name='decoder_lstm1')
    decoder_outputs1 = decoder_layer1(decoder_repeated)
    decoder_bn1 = layers.BatchNormalization()(decoder_outputs1)
    decoder_do1 = layers.Dropout(0.4)(decoder_bn1)
    
    # Decoder Layer 2
    decoder_layer2 = layers.LSTM(512, return_sequences=True, recurrent_dropout=0.3, dropout=0.3, name='decoder_lstm2')
    decoder_outputs2 = decoder_layer2(decoder_do1)
    decoder_bn2 = layers.BatchNormalization()(decoder_outputs2)
    decoder_do2 = layers.Dropout(0.4)(decoder_bn2)
    
    # Decoder Layer 3
    decoder_layer3 = layers.LSTM(256, return_sequences=True, recurrent_dropout=0.3, dropout=0.3, name='decoder_lstm3')
    decoder_outputs3 = decoder_layer3(decoder_do2)
    decoder_bn3 = layers.BatchNormalization()(decoder_outputs3)
    decoder_do3 = layers.Dropout(0.3)(decoder_bn3)
    
    # Dense layers (깊은 구조)
    dense1 = layers.TimeDistributed(layers.Dense(256, activation='relu'))(decoder_do3)
    dense1_bn = layers.TimeDistributed(layers.BatchNormalization())(dense1)
    dense1_do = layers.TimeDistributed(layers.Dropout(0.3))(dense1_bn)
    
    dense2 = layers.TimeDistributed(layers.Dense(128, activation='relu'))(dense1_do)
    dense2_bn = layers.TimeDistributed(layers.BatchNormalization())(dense2)
    dense2_do = layers.TimeDistributed(layers.Dropout(0.3))(dense2_bn)
    
    dense3 = layers.TimeDistributed(layers.Dense(64, activation='relu'))(dense2_do)
    dense3_bn = layers.TimeDistributed(layers.BatchNormalization())(dense3)
    
    dense4 = layers.TimeDistributed(layers.Dense(32, activation='relu'))(dense3_bn)
    
    # Output layer
    outputs = layers.TimeDistributed(layers.Dense(N_OUTPUT_FEATURES))(dense4)
    
    model = Model(encoder_inputs, outputs, name='EncoderOnly_Advanced_Model')
    
    return model

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 모델 3: Autoregressive (심화 버전)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def build_autoregressive_model():
    """
    심화된 Autoregressive 모델
    - Seq2Seq 구조 + 추가 Dense layers
    """
    
    print("\n✅ [Autoregressive] 심화 모델 구축 중...")
    
    # ━━━━━━━ Encoder ━━━━━━━
    encoder_inputs = keras.Input(shape=(SEQUENCE_LENGTH, N_INPUT_FEATURES), name='encoder_input')
    
    # Encoder Layer 1
    encoder_layer1 = layers.Bidirectional(
        layers.LSTM(64, return_sequences=True, recurrent_dropout=0.3, dropout=0.3),
        name='encoder_bilstm1'
    )(encoder_inputs)
    encoder_bn1 = layers.BatchNormalization()(encoder_layer1)
    encoder_do1 = layers.Dropout(0.3)(encoder_bn1)
    
    # Encoder Layer 2
    encoder_layer2 = layers.Bidirectional(
        layers.LSTM(96, return_sequences=True, recurrent_dropout=0.3, dropout=0.3),
        name='encoder_bilstm2'
    )(encoder_do1)
    encoder_bn2 = layers.BatchNormalization()(encoder_layer2)
    encoder_do2 = layers.Dropout(0.3)(encoder_bn2)
    
    # Encoder Layer 3
    encoder_layer3 = layers.Bidirectional(
        layers.LSTM(64, return_state=True, recurrent_dropout=0.3, dropout=0.3),
        name='encoder_bilstm3'
    )
    encoder_outputs, forward_h, forward_c, backward_h, backward_c = encoder_layer3(encoder_do2)
    encoder_bn3 = layers.BatchNormalization()(encoder_outputs)
    
    # Context vector
    state_h = layers.Concatenate()([forward_h, backward_h])
    state_c = layers.Concatenate()([forward_c, backward_c])
    context = layers.Concatenate()([state_h, state_c])
    
    # ━━━━━━━ Decoder ━━━━━━━
    decoder_inputs = keras.Input(shape=(FUTURE_STEPS, N_DECODER_INPUT), name='decoder_input')
    
    # Decoder Layer 1
    decoder_layer1 = layers.LSTM(128, return_sequences=True, recurrent_dropout=0.3, dropout=0.3, name='decoder_lstm1')
    decoder_outputs1 = decoder_layer1(decoder_inputs, initial_state=[state_h, state_c])
    decoder_bn1 = layers.BatchNormalization()(decoder_outputs1)
    decoder_do1 = layers.Dropout(0.3)(decoder_bn1)
    
    # Decoder Layer 2
    decoder_layer2 = layers.LSTM(192, return_sequences=True, recurrent_dropout=0.3, dropout=0.3, name='decoder_lstm2')
    decoder_outputs2 = decoder_layer2(decoder_do1)
    decoder_bn2 = layers.BatchNormalization()(decoder_outputs2)
    decoder_do2 = layers.Dropout(0.3)(decoder_bn2)
    
    # Decoder Layer 3
    decoder_layer3 = layers.LSTM(128, return_sequences=True, recurrent_dropout=0.3, dropout=0.3, name='decoder_lstm3')
    decoder_outputs3 = decoder_layer3(decoder_do2)
    decoder_bn3 = layers.BatchNormalization()(decoder_outputs3)
    decoder_do3 = layers.Dropout(0.3)(decoder_bn3)
    
    # Dense layers
    dense1 = layers.TimeDistributed(layers.Dense(96, activation='relu'))(decoder_do3)
    dense1_bn = layers.TimeDistributed(layers.BatchNormalization())(dense1)
    dense1_do = layers.TimeDistributed(layers.Dropout(0.3))(dense1_bn)
    
    dense2 = layers.TimeDistributed(layers.Dense(64, activation='relu'))(dense1_do)
    dense2_bn = layers.TimeDistributed(layers.BatchNormalization())(dense2)
    dense2_do = layers.TimeDistributed(layers.Dropout(0.2))(dense2_bn)
    
    dense3 = layers.TimeDistributed(layers.Dense(32, activation='relu'))(dense2_do)
    dense3_bn = layers.TimeDistributed(layers.BatchNormalization())(dense3)
    
    # Output layer
    outputs = layers.TimeDistributed(layers.Dense(N_OUTPUT_FEATURES))(dense3_bn)
    
    model = Model([encoder_inputs, decoder_inputs], outputs, name='Autoregressive_Advanced_Model')
    
    return model

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 데이터 로드
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def load_and_prepare_data():
    """데이터 로드 및 분할"""
    
    global X_train, Y_train, X_val, Y_val, X_test, Y_test
    
    print("\n" + "="*80)
    print("데이터 로드 및 준비")
    print("="*80)
    
    X = np.load(f'{DATA_DIR}/X_sequences.npy')
    Y = np.load(f'{DATA_DIR}/Y_sequences.npy')
    
    with open(f'{DATA_DIR}/metadata.pkl', 'rb') as f:
        metadata = pickle.load(f)
    
    print(f"✅ 데이터 로드: X {X.shape}, Y {Y.shape}")
    
    indices = np.arange(len(X))
    train_idx, test_idx = train_test_split(indices, test_size=0.2, random_state=SEED)
    train_idx, val_idx = train_test_split(train_idx, test_size=0.2/(1-0.2), random_state=SEED)
    
    X_train, Y_train = X[train_idx], Y[train_idx]
    X_val, Y_val = X[val_idx], Y[val_idx]
    X_test, Y_test = X[test_idx], Y[test_idx]
    
    print(f"\n데이터 분할:")
    print(f"  Train: {len(X_train):,}")
    print(f"  Val:   {len(X_val):,}")
    print(f"  Test:  {len(X_test):,}")
    
    return (X_train, Y_train), (X_val, Y_val), (X_test, Y_test), metadata

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 모델 학습 함수
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def train_model(model, model_type, train_data, val_data):
    """모델 학습 + 저장"""
    
    global X_train, Y_train, X_val, Y_val
    
    print(f"\n{'='*80}")
    print(f"모델 학습: {model_type.upper()} (심화 버전)")
    print(f"{'='*80}")
    
    X_train_local, Y_train_local = train_data
    X_val_local, Y_val_local = val_data
    
    print(f"\n[1/3] 모델 컴파일 중...")
    optimizer = Adam(learning_rate=0.0001, clipnorm=1.0)
    model.compile(optimizer=optimizer, loss='mse', metrics=['mae'])
    print(f"✅ 컴파일 완료")
    print(f"   총 파라미터: {model.count_params():,}")
    
    print(f"\n[2/3] Callback 설정 중...")
    metrics_callback = MetricsCallback(X_val_local, Y_val_local, model_type=model_type)
    
    callbacks = [
        EarlyStopping(monitor='val_loss', patience=20, restore_best_weights=True, verbose=1),
        ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=10, min_lr=1e-7, verbose=1),
        ModelCheckpoint(f'{MODEL_DIR}/best_{model_type}.keras', monitor='val_loss', save_best_only=True, verbose=0),
        metrics_callback
    ]
    print(f"✅ Callback 설정 완료")
    
    print(f"\n[3/3] 모델 학습 중...")
    
    if model_type in ['seq2seq', 'autoregressive']:
        decoder_inputs_train = Y_train_local[:, :, :-1]
        decoder_inputs_val = Y_val_local[:, :, :-1]
        
        history = model.fit(
            [X_train_local, decoder_inputs_train],
            Y_train_local,
            batch_size=BATCH_SIZE,
            epochs=EPOCHS,
            validation_data=([X_val_local, decoder_inputs_val], Y_val_local),
            callbacks=callbacks,
            verbose=1
        )
    else:
        history = model.fit(
            X_train_local,
            Y_train_local,
            batch_size=BATCH_SIZE,
            epochs=EPOCHS,
            validation_data=(X_val_local, Y_val_local),
            callbacks=callbacks,
            verbose=1
        )
    
    print(f"\n✅ 학습 완료")
    
    print(f"\n[4/3] 모델 저장 중 (.keras 포맷)...", end='', flush=True)
    model.save(f'{MODEL_DIR}/{model_type}_model.keras', save_format='keras')
    print(" ✅")
    print(f"  저장 위치: {MODEL_DIR}/{model_type}_model.keras")
    
    return history, metrics_callback

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 모델 평가 함수
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def evaluate_model(model, model_type, test_data):
    """모델 평가"""
    
    print(f"\n{'='*80}")
    print(f"모델 평가: {model_type.upper()}")
    print(f"{'='*80}")
    
    X_test_local, Y_test_local = test_data
    total_samples = len(X_test_local)
    
    print(f"\n[1/2] 예측 중 [0/{total_samples}]...", end='\r', flush=True)
    
    if model_type == 'seq2seq':
        decoder_input = Y_test_local[:, :, :-1]
        predictions = model.predict([X_test_local, decoder_input], batch_size=32, verbose=0)
    elif model_type == 'encoder_only':
        predictions = model.predict(X_test_local, batch_size=32, verbose=0)
    else:  # autoregressive
        print(f"\n[1/2] Autoregressive 예측 중 (배치 처리)...")
        predictions = autoregressive_predict_batch_fast(model, X_test_local, batch_size=64)
    
    print(f"[1/2] 예측 완료 [{total_samples}/{total_samples}] ✅")
    
    print(f"\n[2/2] 평가 중...", end='', flush=True)
    
    pred_features = predictions[:, :, :-1]
    target_features = Y_test_local[:, :, :-1]
    
    pred_outcome_prob = predictions[:, :, -1].flatten()
    pred_outcome = (pred_outcome_prob > 0.5).astype(int)
    target_outcome = Y_test_local[:, :, -1].astype(int).flatten()
    
    r2_scores = []
    for feat_idx in range(N_DECODER_INPUT):
        r2 = r2_score(
            target_features[:, :, feat_idx].flatten(),
            pred_features[:, :, feat_idx].flatten()
        )
        r2_scores.append(r2)
    
    r2 = np.mean(r2_scores)
    mae = np.mean(np.abs(target_features - pred_features))
    mse = np.mean((target_features - pred_features) ** 2)
    rmse = np.sqrt(mse)
    
    outcome_acc = accuracy_score(target_outcome, pred_outcome)
    outcome_f1 = f1_score(target_outcome, pred_outcome, zero_division=0)
    outcome_auc = roc_auc_score(target_outcome, pred_outcome_prob)
    
    fpr, tpr, thresholds = roc_curve(target_outcome, pred_outcome_prob)
    cm = confusion_matrix(target_outcome, pred_outcome)
    
    print(" ✅")
    
    print(f"\n📊 성능 지표 (Test):")
    print(f"  R² Score: {r2:.4f}")
    print(f"  MAE: {mae:.6f}")
    print(f"  RMSE: {rmse:.6f}")
    print(f"  Outcome Accuracy: {outcome_acc:.4f} ({outcome_acc*100:.2f}%)")
    print(f"  Outcome F1-Score: {outcome_f1:.4f}")
    print(f"  Outcome AUC-ROC: {outcome_auc:.4f}")
    
    print(f"\n📈 Confusion Matrix:")
    print(f"  True Negatives: {cm[0, 0]}, False Positives: {cm[0, 1]}")
    print(f"  False Negatives: {cm[1, 0]}, True Positives: {cm[1, 1]}")
    
    return {
        'r2': r2,
        'mae': mae,
        'mse': mse,
        'rmse': rmse,
        'outcome_accuracy': outcome_acc,
        'outcome_f1': outcome_f1,
        'outcome_auc': outcome_auc,
        'fpr': fpr,
        'tpr': tpr,
        'threshold': thresholds,
        'pred_outcome_prob': pred_outcome_prob,
        'target_outcome': target_outcome,
        'confusion_matrix': cm,
    }

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 시각화 함수
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def plot_comparison(results_dict):
    """세 모델 비교 시각화"""
    
    print(f"\n{'='*80}")
    print("시각화: 세 모델 비교")
    print(f"{'='*80}")
    
    os.makedirs(f'{MODEL_DIR}/plots', exist_ok=True)
    
    seq2seq_results = results_dict['seq2seq']
    encoder_results = results_dict['encoder_only']
    autoregressive_results = results_dict['autoregressive']
    
    fig, axes = plt.subplots(2, 2, figsize=(15, 12))
    fig.suptitle('Seq2Seq vs Encoder-only vs Autoregressive (심화 모델)', fontsize=16, fontweight='bold')
    
    # ROC Curve
    ax = axes[0, 0]
    ax.plot([0, 1], [0, 1], 'k--', label='Random Classifier', linewidth=2)
    ax.plot(seq2seq_results['fpr'], seq2seq_results['tpr'], 
            label=f"Seq2Seq (AUC = {seq2seq_results['outcome_auc']:.4f})",
            linewidth=2.5, marker='o', markersize=4)
    ax.plot(encoder_results['fpr'], encoder_results['tpr'],
            label=f"Encoder-only (AUC = {encoder_results['outcome_auc']:.4f})",
            linewidth=2.5, marker='s', markersize=4)
    ax.plot(autoregressive_results['fpr'], autoregressive_results['tpr'],
            label=f"Autoregressive (AUC = {autoregressive_results['outcome_auc']:.4f})",
            linewidth=2.5, marker='^', markersize=4)
    ax.set_xlabel('False Positive Rate', fontsize=11)
    ax.set_ylabel('True Positive Rate', fontsize=11)
    ax.set_title('ROC Curve 비교', fontsize=12, fontweight='bold')
    ax.legend(loc='lower right', fontsize=10)
    ax.grid(True, alpha=0.3)
    
    # 성능 지표 비교
    ax = axes[0, 1]
    metrics = ['R²', 'Accuracy', 'F1-Score', 'AUC']
    seq2seq_vals = [
        seq2seq_results['r2'],
        seq2seq_results['outcome_accuracy'],
        seq2seq_results['outcome_f1'],
        seq2seq_results['outcome_auc']
    ]
    encoder_vals = [
        encoder_results['r2'],
        encoder_results['outcome_accuracy'],
        encoder_results['outcome_f1'],
        encoder_results['outcome_auc']
    ]
    autoregressive_vals = [
        autoregressive_results['r2'],
        autoregressive_results['outcome_accuracy'],
        autoregressive_results['outcome_f1'],
        autoregressive_results['outcome_auc']
    ]
    
    x = np.arange(len(metrics))
    width = 0.25
    ax.bar(x - width, seq2seq_vals, width, label='Seq2Seq', alpha=0.8)
    ax.bar(x, encoder_vals, width, label='Encoder-only', alpha=0.8)
    ax.bar(x + width, autoregressive_vals, width, label='Autoregressive', alpha=0.8)
    ax.set_ylabel('Score', fontsize=11)
    ax.set_title('성능 지표 비교', fontsize=12, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(metrics)
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3, axis='y')
    ax.set_ylim([0, 1])
    
    # Confusion Matrix
    ax = axes[1, 0]
    cm_combined = np.array([
        seq2seq_results['confusion_matrix'].flatten(),
        encoder_results['confusion_matrix'].flatten(),
        autoregressive_results['confusion_matrix'].flatten()
    ])
    im = ax.imshow(cm_combined, cmap='Blues', aspect='auto')
    ax.set_xticks([0, 1, 2, 3])
    ax.set_xticklabels(['TN', 'FP', 'FN', 'TP'])
    ax.set_yticks([0, 1, 2])
    ax.set_yticklabels(['Seq2Seq', 'Encoder-only', 'Autoregressive'])
    ax.set_title('Confusion Matrix 비교', fontsize=12, fontweight='bold')
    
    for i in range(3):
        for j in range(4):
            text = ax.text(j, i, f'{cm_combined[i, j]:.0f}',
                          ha="center", va="center", color="w", fontsize=10)
    
    # MAE 비교
    ax = axes[1, 1]
    models = ['Seq2Seq', 'Encoder-only', 'Autoregressive']
    mae_vals = [seq2seq_results['mae'], encoder_results['mae'], autoregressive_results['mae']]
    colors = ['#1f77b4', '#ff7f0e', '#2ca02c']
    bars = ax.bar(models, mae_vals, color=colors, alpha=0.8, edgecolor='black', linewidth=1.5)
    ax.set_ylabel('MAE', fontsize=11)
    ax.set_title('평균 절대 오차 (MAE) 비교', fontsize=12, fontweight='bold')
    ax.grid(True, alpha=0.3, axis='y')
    
    for bar, val in zip(bars, mae_vals):
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
                f'{val:.6f}', ha='center', va='bottom', fontsize=10)
    
    plt.tight_layout()
    plt.savefig(f'{MODEL_DIR}/plots/model_comparison_3way.png', dpi=300, bbox_inches='tight')
    print(f"✅ 저장: {MODEL_DIR}/plots/model_comparison_3way.png")
    plt.close()

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 메인 함수
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def main():
    """메인 함수"""
    
    global seq2seq_model, encoder_model, autoregressive_model
    global seq2seq_results, encoder_results, autoregressive_results
    
    print("\n" + "="*80)
    print("Valorant 라운드 예측: 세 모델 비교 (심화 아키텍처 버전)")
    print("="*80)
    print("\n개선사항:")
    print("- BatchNormalization: 모든 LSTM/Dense 후 적용")
    print("- Dropout: 0.2~0.4 범위로 적절히 배치")
    print("- Dense Layers: 3~4단계 깊은 구조")
    print("- LayerNormalization: LSTM 출력 안정화")
    print("- Multi-layer LSTM: 3단계 아키텍처")
    print("="*80)
    
    os.makedirs(MODEL_DIR, exist_ok=True)
    
    # Step 1: 데이터 로드
    train_data, val_data, test_data, metadata = load_and_prepare_data()
    
    # Step 2: 모델 구축 (심화 버전)
    print(f"\n{'='*80}")
    print("모델 구축 (심화 아키텍처)")
    print(f"{'='*80}")
    
    seq2seq_model = build_seq2seq_model()
    encoder_model = build_encoder_only_model()
    autoregressive_model = build_autoregressive_model()
    
    print(f"\n✅ 세 모델 구축 완료")
    
    # Step 3: 모델 학습
    print(f"\n{'='*80}")
    print("모델 학습")
    print(f"{'='*80}")
    
    seq2seq_history, seq2seq_metrics_callback = train_model(
        seq2seq_model, 'seq2seq', train_data, val_data
    )
    
    print(f"\n" + "-"*80 + "\n")
    
    encoder_history, encoder_metrics_callback = train_model(
        encoder_model, 'encoder_only', train_data, val_data
    )
    
    print(f"\n" + "-"*80 + "\n")
    
    autoregressive_history, autoregressive_metrics_callback = train_model(
        autoregressive_model, 'autoregressive', train_data, val_data
    )
    
    # Step 4: 모델 평가
    print(f"\n{'='*80}")
    print("모델 평가")
    print(f"{'='*80}")
    
    seq2seq_results = evaluate_model(seq2seq_model, 'seq2seq', test_data)
    encoder_results = evaluate_model(encoder_model, 'encoder_only', test_data)
    autoregressive_results = evaluate_model(autoregressive_model, 'autoregressive', test_data)
    
    # Step 5: 최종 비교
    print(f"\n{'='*80}")
    print("최종 비교 결과")
    print(f"{'='*80}")
    
    print(f"\n📊 성능 지표 비교 (Test 세트):")
    print(f"\n{'지표':<30} {'Seq2Seq':>15} {'Encoder-only':>15} {'Autoregressive':>15}")
    print(f"{'-'*75}")
    
    comparison_data = [
        ('R² Score', seq2seq_results['r2'], encoder_results['r2'], autoregressive_results['r2']),
        ('MAE', seq2seq_results['mae'], encoder_results['mae'], autoregressive_results['mae']),
        ('RMSE', seq2seq_results['rmse'], encoder_results['rmse'], autoregressive_results['rmse']),
        ('Outcome Accuracy', seq2seq_results['outcome_accuracy'], encoder_results['outcome_accuracy'], autoregressive_results['outcome_accuracy']),
        ('F1-Score', seq2seq_results['outcome_f1'], encoder_results['outcome_f1'], autoregressive_results['outcome_f1']),
        ('AUC-ROC', seq2seq_results['outcome_auc'], encoder_results['outcome_auc'], autoregressive_results['outcome_auc']),
    ]
    
    for metric_name, seq2seq_val, encoder_val, autoregressive_val in comparison_data:
        print(f"{metric_name:<30} {seq2seq_val:>15.4f} {encoder_val:>15.4f} {autoregressive_val:>15.4f}")
    
    # Step 6: 시각화
    print(f"\n{'='*80}")
    print("시각화")
    print(f"{'='*80}")
    
    results_dict = {
        'seq2seq': seq2seq_results,
        'encoder_only': encoder_results,
        'autoregressive': autoregressive_results
    }
    
    plot_comparison(results_dict)
    
    # Step 7: 최종 저장
    print(f"\n{'='*80}")
    print("최종 결과 저장")
    print(f"{'='*80}")
    
    final_results = {
        'seq2seq_results': seq2seq_results,
        'encoder_only_results': encoder_results,
        'autoregressive_results': autoregressive_results,
        'comparison_data': comparison_data,
    }
    
    with open(f'{MODEL_DIR}/final_results_3way.pkl', 'wb') as f:
        pickle.dump(final_results, f)
    
    print(f"\n✅ 저장 완료:")
    print(f"  - {MODEL_DIR}/seq2seq_model.keras")
    print(f"  - {MODEL_DIR}/encoder_only_model.keras")
    print(f"  - {MODEL_DIR}/autoregressive_model.keras")
    print(f"  - {MODEL_DIR}/final_results_3way.pkl")
    print(f"  - {MODEL_DIR}/plots/model_comparison_3way.png")
    
    print(f"\n{'='*80}")
    print("✅ 전체 파이프라인 완료!")
    print(f"{'='*80}")

if __name__ == "__main__":
    main()

In [ ]:
"""
Valorant 라운드 예측 - Encoder-only & Autoregressive 개선 버전
================================================================

개선사항:
1. Encoder-only: Time-distributed Dense로 시간 정보 다양화
2. Autoregressive: 진짜 순차 생성 (한 스텝씩 피드백)

목표:
- Encoder-only: R² 0.35 → 0.55+, AUC 0.64 → 0.75+
- Autoregressive: R² 0.17 → 0.60+, AUC 0.63 → 0.80+
"""

import numpy as np
import pandas as pd
import os
import pickle
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    mean_squared_error, r2_score, accuracy_score, f1_score,
    roc_curve, auc, roc_auc_score, confusion_matrix, classification_report
)
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model
from tensorflow.keras.callbacks import Callback, EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.optimizers import Adam
import warnings
warnings.filterwarnings('ignore')

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 전역 변수
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

encoder_model = None
autoregressive_model = None

encoder_results = None
autoregressive_results = None

X_train = None
Y_train = None
X_val = None
Y_val = None
X_test = None
Y_test = None

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 설정
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

DATA_DIR = './data/seq2seq_data'
MODEL_DIR = './models/seq2seq_comparison'
BATCH_SIZE = 32  # ✅ 증가 (16 → 32)
EPOCHS = 100
SEED = 42

SEQUENCE_LENGTH = 12
FUTURE_STEPS = 12
N_INPUT_FEATURES = 8
N_OUTPUT_FEATURES = 9
N_DECODER_INPUT = 8

np.random.seed(SEED)
tf.random.set_seed(SEED)

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# Callback: 상세 지표 계산
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

class MetricsCallback(Callback):
    """매 에포크마다 상세 지표 계산"""
    
    def __init__(self, X_val, Y_val, model_type='encoder_only'):
        super().__init__()
        self.X_val = X_val
        self.Y_val = Y_val
        self.model_type = model_type
        self.history = {
            'val_r2': [],
            'val_mae': [],
            'val_outcome_acc': [],
            'val_outcome_f1': [],
            'val_auc': [],
        }
    
    def on_epoch_end(self, epoch, logs=None):
        """에포크 끝날 때마다 호출"""
        
        if self.model_type == 'encoder_only':
            val_pred = self.model.predict(self.X_val, verbose=0)
        else:  # autoregressive - 진짜 순차 생성
            val_pred = autoregressive_predict_real(self.model, self.X_val)
        
        val_pred_features = val_pred[:, :, :-1]
        val_targets_features = self.Y_val[:, :, :-1]
        
        val_pred_outcome_prob = val_pred[:, :, -1].flatten()
        val_pred_outcome = (val_pred_outcome_prob > 0.5).astype(int)
        val_targets_outcome = self.Y_val[:, :, -1].astype(int).flatten()
        
        try:
            r2_scores = []
            for feat_idx in range(N_DECODER_INPUT):
                r2 = r2_score(
                    val_targets_features[:, :, feat_idx].flatten(),
                    val_pred_features[:, :, feat_idx].flatten()
                )
                r2_scores.append(r2)
            val_r2 = np.mean(r2_scores)
        except:
            val_r2 = -1.0
        
        val_mae = np.mean(np.abs(val_targets_features - val_pred_features))
        val_outcome_acc = accuracy_score(val_targets_outcome, val_pred_outcome)
        val_outcome_f1 = f1_score(val_targets_outcome, val_pred_outcome, zero_division=0)
        
        try:
            val_auc = roc_auc_score(val_targets_outcome, val_pred_outcome_prob)
        except:
            val_auc = 0.5
        
        self.history['val_r2'].append(val_r2)
        self.history['val_mae'].append(val_mae)
        self.history['val_outcome_acc'].append(val_outcome_acc)
        self.history['val_outcome_f1'].append(val_outcome_f1)
        self.history['val_auc'].append(val_auc)
        
        if (epoch + 1) % 20 == 0:
            print(f"\n[Epoch {epoch+1}] {self.model_type.upper()} Validation:")
            print(f"  R²: {val_r2:.4f}, MAE: {val_mae:.6f}, AUC: {val_auc:.4f}")
            print(f"  Outcome Acc: {val_outcome_acc:.4f}, F1: {val_outcome_f1:.4f}")

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# Autoregressive 진짜 순차 생성 (개선)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def autoregressive_predict_real(model, X_batch, batch_size=32):
    """
    ✅ 진짜 Autoregressive 예측 (한 스텝씩 순차 생성)
    
    이전 예측을 다음 입력으로 사용 (피드백)
    """
    
    total_samples = len(X_batch)
    predictions = np.zeros((total_samples, FUTURE_STEPS, N_OUTPUT_FEATURES), dtype=np.float32)
    
    for batch_start in range(0, total_samples, batch_size):
        batch_end = min(batch_start + batch_size, total_samples)
        X_batch_slice = X_batch[batch_start:batch_end]
        batch_len = len(X_batch_slice)
        
        # Decoder 입력 초기화 (0으로 시작)
        decoder_seq = np.zeros((batch_len, FUTURE_STEPS, N_DECODER_INPUT), dtype=np.float32)
        
        # ✅ 한 스텝씩 순차 생성
        for step in range(FUTURE_STEPS):
            # 현재까지의 decoder 입력으로 예측
            pred = model.predict([X_batch_slice, decoder_seq], verbose=0)
            
            # step번째 라운드 예측 저장
            predictions[batch_start:batch_end, step, :] = pred[:, step, :]
            
            # ✅ 다음 스텝을 위해 현재 예측을 decoder 입력으로 피드백
            if step < FUTURE_STEPS - 1:
                decoder_seq[:, step + 1, :] = pred[:, step, :N_DECODER_INPUT]
        
        percent = (batch_end / total_samples) * 100
        print(f"    [{batch_end}/{total_samples}] {percent:.1f}%", end='\r', flush=True)
    
    print(f"    [{total_samples}/{total_samples}] 100.0%")
    
    return predictions

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 모델 1: Encoder-only (개선 버전 - Time-distributed Dense로 다양화)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def build_encoder_only_improved():
    """
    ✅ 개선된 Encoder-only 모델
    
    핵심 개선:
    - RepeatVector 후 Time-distributed Dense로 각 타임스텝 다양화
    - 더 큰 LSTM 유닛 (128-256-128)
    - Dropout 감소 (0.3 → 0.2)
    """
    
    print("\n✅ [Encoder-only] 개선 모델 구축 중...")
    
    # ━━━━━━━ Encoder ━━━━━━━
    encoder_inputs = keras.Input(shape=(SEQUENCE_LENGTH, N_INPUT_FEATURES), name='encoder_input')
    
    # Encoder Layer 1
    encoder_layer1 = layers.Bidirectional(
        layers.LSTM(128, return_sequences=True, recurrent_dropout=0.2, dropout=0.2),
        name='encoder_bilstm1'
    )(encoder_inputs)
    encoder_bn1 = layers.BatchNormalization()(encoder_layer1)
    encoder_do1 = layers.Dropout(0.2)(encoder_bn1)
    
    # Encoder Layer 2
    encoder_layer2 = layers.Bidirectional(
        layers.LSTM(256, return_sequences=True, recurrent_dropout=0.2, dropout=0.2),
        name='encoder_bilstm2'
    )(encoder_do1)
    encoder_bn2 = layers.BatchNormalization()(encoder_layer2)
    encoder_do2 = layers.Dropout(0.2)(encoder_bn2)
    
    # Encoder Layer 3
    encoder_layer3 = layers.Bidirectional(
        layers.LSTM(128, return_state=True, recurrent_dropout=0.2, dropout=0.2),
        name='encoder_bilstm3'
    )
    encoder_outputs, forward_h, forward_c, backward_h, backward_c = encoder_layer3(encoder_do2)
    encoder_bn3 = layers.BatchNormalization()(encoder_outputs)
    
    # Context vector (더 큰 용량)
    state_h = layers.Concatenate()([forward_h, backward_h])
    state_c = layers.Concatenate()([forward_c, backward_c])
    
    context_1 = layers.Concatenate()([state_h, state_c])
    context_dense1 = layers.Dense(512, activation='relu')(context_1)
    context_bn1 = layers.BatchNormalization()(context_dense1)
    context_do1 = layers.Dropout(0.3)(context_bn1)
    
    context_dense2 = layers.Dense(384, activation='relu')(context_do1)
    context_bn2 = layers.BatchNormalization()(context_dense2)
    context_do2 = layers.Dropout(0.3)(context_bn2)
    
    context_final = layers.Dense(256, activation='relu')(context_do2)
    context_final_bn = layers.BatchNormalization()(context_final)
    
    # ━━━━━━━ Decoder (개선: Time-distributed Dense로 다양화) ━━━━━━━
    # RepeatVector로 복사
    decoder_repeated = layers.RepeatVector(FUTURE_STEPS)(context_final_bn)
    
    # ✅ Time-distributed Dense로 각 타임스텝을 다르게 변환
    time_diverse = layers.TimeDistributed(
        layers.Dense(256, activation='tanh'),
        name='time_diversification'
    )(decoder_repeated)
    time_diverse_bn = layers.TimeDistributed(layers.BatchNormalization())(time_diverse)
    
    # Decoder Layer 1
    decoder_layer1 = layers.LSTM(256, return_sequences=True, recurrent_dropout=0.2, dropout=0.2, name='decoder_lstm1')
    decoder_outputs1 = decoder_layer1(time_diverse_bn)
    decoder_bn1 = layers.BatchNormalization()(decoder_outputs1)
    decoder_do1 = layers.Dropout(0.3)(decoder_bn1)
    
    # Decoder Layer 2
    decoder_layer2 = layers.LSTM(384, return_sequences=True, recurrent_dropout=0.2, dropout=0.2, name='decoder_lstm2')
    decoder_outputs2 = decoder_layer2(decoder_do1)
    decoder_bn2 = layers.BatchNormalization()(decoder_outputs2)
    decoder_do2 = layers.Dropout(0.3)(decoder_bn2)
    
    # Decoder Layer 3
    decoder_layer3 = layers.LSTM(256, return_sequences=True, recurrent_dropout=0.2, dropout=0.2, name='decoder_lstm3')
    decoder_outputs3 = decoder_layer3(decoder_do2)
    decoder_bn3 = layers.BatchNormalization()(decoder_outputs3)
    decoder_do3 = layers.Dropout(0.2)(decoder_bn3)
    
    # Dense layers
    dense1 = layers.TimeDistributed(layers.Dense(192, activation='relu'))(decoder_do3)
    dense1_bn = layers.TimeDistributed(layers.BatchNormalization())(dense1)
    dense1_do = layers.TimeDistributed(layers.Dropout(0.2))(dense1_bn)
    
    dense2 = layers.TimeDistributed(layers.Dense(96, activation='relu'))(dense1_do)
    dense2_bn = layers.TimeDistributed(layers.BatchNormalization())(dense2)
    dense2_do = layers.TimeDistributed(layers.Dropout(0.2))(dense2_bn)
    
    dense3 = layers.TimeDistributed(layers.Dense(48, activation='relu'))(dense2_do)
    dense3_bn = layers.TimeDistributed(layers.BatchNormalization())(dense3)
    
    # Output layer
    outputs = layers.TimeDistributed(layers.Dense(N_OUTPUT_FEATURES))(dense3_bn)
    
    model = Model(encoder_inputs, outputs, name='EncoderOnly_Improved_Model')
    
    return model

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 모델 2: Autoregressive (개선 버전 - 순차 생성 준비)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def build_autoregressive_improved():
    """
    ✅ 개선된 Autoregressive 모델
    
    핵심 개선:
    - 더 큰 LSTM 유닛 (128-192-128)
    - Dropout 감소 (0.3 → 0.2)
    - 평가 시 진짜 순차 생성 사용
    """
    
    print("\n✅ [Autoregressive] 개선 모델 구축 중...")
    
    # ━━━━━━━ Encoder ━━━━━━━
    encoder_inputs = keras.Input(shape=(SEQUENCE_LENGTH, N_INPUT_FEATURES), name='encoder_input')
    
    # Encoder Layer 1
    encoder_layer1 = layers.Bidirectional(
        layers.LSTM(128, return_sequences=True, recurrent_dropout=0.2, dropout=0.2),
        name='encoder_bilstm1'
    )(encoder_inputs)
    encoder_bn1 = layers.BatchNormalization()(encoder_layer1)
    encoder_do1 = layers.Dropout(0.2)(encoder_bn1)
    
    # Encoder Layer 2
    encoder_layer2 = layers.Bidirectional(
        layers.LSTM(192, return_sequences=True, recurrent_dropout=0.2, dropout=0.2),
        name='encoder_bilstm2'
    )(encoder_do1)
    encoder_bn2 = layers.BatchNormalization()(encoder_layer2)
    encoder_do2 = layers.Dropout(0.2)(encoder_bn2)
    
    # Encoder Layer 3
    encoder_layer3 = layers.Bidirectional(
        layers.LSTM(128, return_state=True, recurrent_dropout=0.2, dropout=0.2),
        name='encoder_bilstm3'
    )
    encoder_outputs, forward_h, forward_c, backward_h, backward_c = encoder_layer3(encoder_do2)
    encoder_bn3 = layers.BatchNormalization()(encoder_outputs)
    
    # Context vector
    state_h = layers.Concatenate()([forward_h, backward_h])
    state_c = layers.Concatenate()([forward_c, backward_c])
    
    # ━━━━━━━ Decoder ━━━━━━━
    decoder_inputs = keras.Input(shape=(FUTURE_STEPS, N_DECODER_INPUT), name='decoder_input')
    
    # Decoder Layer 1
    decoder_layer1 = layers.LSTM(192, return_sequences=True, recurrent_dropout=0.2, dropout=0.2, name='decoder_lstm1')
    decoder_outputs1 = decoder_layer1(decoder_inputs, initial_state=[state_h, state_c])
    decoder_bn1 = layers.BatchNormalization()(decoder_outputs1)
    decoder_do1 = layers.Dropout(0.2)(decoder_bn1)
    
    # Decoder Layer 2
    decoder_layer2 = layers.LSTM(256, return_sequences=True, recurrent_dropout=0.2, dropout=0.2, name='decoder_lstm2')
    decoder_outputs2 = decoder_layer2(decoder_do1)
    decoder_bn2 = layers.BatchNormalization()(decoder_outputs2)
    decoder_do2 = layers.Dropout(0.2)(decoder_bn2)
    
    # Decoder Layer 3
    decoder_layer3 = layers.LSTM(192, return_sequences=True, recurrent_dropout=0.2, dropout=0.2, name='decoder_lstm3')
    decoder_outputs3 = decoder_layer3(decoder_do2)
    decoder_bn3 = layers.BatchNormalization()(decoder_outputs3)
    decoder_do3 = layers.Dropout(0.2)(decoder_bn3)
    
    # Dense layers
    dense1 = layers.TimeDistributed(layers.Dense(128, activation='relu'))(decoder_do3)
    dense1_bn = layers.TimeDistributed(layers.BatchNormalization())(dense1)
    dense1_do = layers.TimeDistributed(layers.Dropout(0.2))(dense1_bn)
    
    dense2 = layers.TimeDistributed(layers.Dense(64, activation='relu'))(dense1_do)
    dense2_bn = layers.TimeDistributed(layers.BatchNormalization())(dense2)
    dense2_do = layers.TimeDistributed(layers.Dropout(0.15))(dense2_bn)
    
    dense3 = layers.TimeDistributed(layers.Dense(32, activation='relu'))(dense2_do)
    dense3_bn = layers.TimeDistributed(layers.BatchNormalization())(dense3)
    
    # Output layer
    outputs = layers.TimeDistributed(layers.Dense(N_OUTPUT_FEATURES))(dense3_bn)
    
    model = Model([encoder_inputs, decoder_inputs], outputs, name='Autoregressive_Improved_Model')
    
    return model

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 데이터 로드
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def load_and_prepare_data():
    """데이터 로드 및 분할"""
    
    global X_train, Y_train, X_val, Y_val, X_test, Y_test
    
    print("\n" + "="*80)
    print("데이터 로드 및 준비")
    print("="*80)
    
    X = np.load(f'{DATA_DIR}/X_sequences.npy')
    Y = np.load(f'{DATA_DIR}/Y_sequences.npy')
    
    with open(f'{DATA_DIR}/metadata.pkl', 'rb') as f:
        metadata = pickle.load(f)
    
    print(f"✅ 데이터 로드: X {X.shape}, Y {Y.shape}")
    
    indices = np.arange(len(X))
    train_idx, test_idx = train_test_split(indices, test_size=0.2, random_state=SEED)
    train_idx, val_idx = train_test_split(train_idx, test_size=0.2/(1-0.2), random_state=SEED)
    
    X_train, Y_train = X[train_idx], Y[train_idx]
    X_val, Y_val = X[val_idx], Y[val_idx]
    X_test, Y_test = X[test_idx], Y[test_idx]
    
    print(f"\n데이터 분할:")
    print(f"  Train: {len(X_train):,}")
    print(f"  Val:   {len(X_val):,}")
    print(f"  Test:  {len(X_test):,}")
    
    return (X_train, Y_train), (X_val, Y_val), (X_test, Y_test), metadata

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 모델 학습 함수
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def train_model(model, model_type, train_data, val_data):
    """모델 학습 + 저장"""
    
    global X_train, Y_train, X_val, Y_val
    
    print(f"\n{'='*80}")
    print(f"모델 학습: {model_type.upper()} (개선 버전)")
    print(f"{'='*80}")
    
    X_train_local, Y_train_local = train_data
    X_val_local, Y_val_local = val_data
    
    print(f"\n[1/3] 모델 컴파일 중...")
    # ✅ Learning Rate 약간 증가 (0.0001 → 0.0003)
    optimizer = Adam(learning_rate=0.0003, clipnorm=1.0)
    model.compile(optimizer=optimizer, loss='mse', metrics=['mae'])
    print(f"✅ 컴파일 완료")
    print(f"   총 파라미터: {model.count_params():,}")
    
    print(f"\n[2/3] Callback 설정 중...")
    metrics_callback = MetricsCallback(X_val_local, Y_val_local, model_type=model_type)
    
    callbacks = [
        EarlyStopping(monitor='val_loss', patience=20, restore_best_weights=True, verbose=1),
        ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=10, min_lr=1e-7, verbose=1),
        ModelCheckpoint(f'{MODEL_DIR}/best_{model_type}_improved.keras', monitor='val_loss', save_best_only=True, verbose=0),
        metrics_callback
    ]
    print(f"✅ Callback 설정 완료")
    
    print(f"\n[3/3] 모델 학습 중...")
    
    if model_type == 'autoregressive':
        decoder_inputs_train = Y_train_local[:, :, :-1]
        decoder_inputs_val = Y_val_local[:, :, :-1]
        
        history = model.fit(
            [X_train_local, decoder_inputs_train],
            Y_train_local,
            batch_size=BATCH_SIZE,
            epochs=EPOCHS,
            validation_data=([X_val_local, decoder_inputs_val], Y_val_local),
            callbacks=callbacks,
            verbose=1
        )
    else:  # encoder_only
        history = model.fit(
            X_train_local,
            Y_train_local,
            batch_size=BATCH_SIZE,
            epochs=EPOCHS,
            validation_data=(X_val_local, Y_val_local),
            callbacks=callbacks,
            verbose=1
        )
    
    print(f"\n✅ 학습 완료")
    
    print(f"\n[4/3] 모델 저장 중 (.keras 포맷)...", end='', flush=True)
    model.save(f'{MODEL_DIR}/{model_type}_model_improved.keras', save_format='keras')
    print(" ✅")
    print(f"  저장 위치: {MODEL_DIR}/{model_type}_model_improved.keras")
    
    return history, metrics_callback

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 모델 평가 함수
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def evaluate_model(model, model_type, test_data):
    """모델 평가"""
    
    print(f"\n{'='*80}")
    print(f"모델 평가: {model_type.upper()}")
    print(f"{'='*80}")
    
    X_test_local, Y_test_local = test_data
    total_samples = len(X_test_local)
    
    print(f"\n[1/2] 예측 중 [0/{total_samples}]...", end='\r', flush=True)
    
    if model_type == 'encoder_only':
        predictions = model.predict(X_test_local, batch_size=32, verbose=0)
        print(f"[1/2] 예측 완료 [{total_samples}/{total_samples}] ✅")
    else:  # autoregressive - 진짜 순차 생성
        print(f"\n[1/2] Autoregressive 예측 중 (순차 생성)...")
        predictions = autoregressive_predict_real(model, X_test_local, batch_size=32)
    
    print(f"\n[2/2] 평가 중...", end='', flush=True)
    
    pred_features = predictions[:, :, :-1]
    target_features = Y_test_local[:, :, :-1]
    
    pred_outcome_prob = predictions[:, :, -1].flatten()
    pred_outcome = (pred_outcome_prob > 0.5).astype(int)
    target_outcome = Y_test_local[:, :, -1].astype(int).flatten()
    
    r2_scores = []
    for feat_idx in range(N_DECODER_INPUT):
        r2 = r2_score(
            target_features[:, :, feat_idx].flatten(),
            pred_features[:, :, feat_idx].flatten()
        )
        r2_scores.append(r2)
    
    r2 = np.mean(r2_scores)
    mae = np.mean(np.abs(target_features - pred_features))
    mse = np.mean((target_features - pred_features) ** 2)
    rmse = np.sqrt(mse)
    
    outcome_acc = accuracy_score(target_outcome, pred_outcome)
    outcome_f1 = f1_score(target_outcome, pred_outcome, zero_division=0)
    outcome_auc = roc_auc_score(target_outcome, pred_outcome_prob)
    
    fpr, tpr, thresholds = roc_curve(target_outcome, pred_outcome_prob)
    cm = confusion_matrix(target_outcome, pred_outcome)
    
    print(" ✅")
    
    print(f"\n📊 성능 지표 (Test):")
    print(f"  R² Score: {r2:.4f}")
    print(f"  MAE: {mae:.6f}")
    print(f"  RMSE: {rmse:.6f}")
    print(f"  Outcome Accuracy: {outcome_acc:.4f} ({outcome_acc*100:.2f}%)")
    print(f"  Outcome F1-Score: {outcome_f1:.4f}")
    print(f"  Outcome AUC-ROC: {outcome_auc:.4f}")
    
    print(f"\n📈 Confusion Matrix:")
    print(f"  True Negatives: {cm[0, 0]}, False Positives: {cm[0, 1]}")
    print(f"  False Negatives: {cm[1, 0]}, True Positives: {cm[1, 1]}")
    
    return {
        'r2': r2,
        'mae': mae,
        'mse': mse,
        'rmse': rmse,
        'outcome_accuracy': outcome_acc,
        'outcome_f1': outcome_f1,
        'outcome_auc': outcome_auc,
        'fpr': fpr,
        'tpr': tpr,
        'threshold': thresholds,
        'pred_outcome_prob': pred_outcome_prob,
        'target_outcome': target_outcome,
        'confusion_matrix': cm,
    }

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 시각화 함수
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def plot_comparison(encoder_results, autoregressive_results):
    """두 모델 비교 시각화"""
    
    print(f"\n{'='*80}")
    print("시각화: Encoder-only vs Autoregressive (개선 버전)")
    print(f"{'='*80}")
    
    os.makedirs(f'{MODEL_DIR}/plots', exist_ok=True)
    
    fig, axes = plt.subplots(2, 2, figsize=(15, 12))
    fig.suptitle('Encoder-only vs Autoregressive (개선 버전)', fontsize=16, fontweight='bold')
    
    # ROC Curve
    ax = axes[0, 0]
    ax.plot([0, 1], [0, 1], 'k--', label='Random Classifier', linewidth=2)
    ax.plot(encoder_results['fpr'], encoder_results['tpr'],
            label=f"Encoder-only (AUC = {encoder_results['outcome_auc']:.4f})",
            linewidth=2.5, marker='s', markersize=4)
    ax.plot(autoregressive_results['fpr'], autoregressive_results['tpr'],
            label=f"Autoregressive (AUC = {autoregressive_results['outcome_auc']:.4f})",
            linewidth=2.5, marker='^', markersize=4)
    ax.set_xlabel('False Positive Rate', fontsize=11)
    ax.set_ylabel('True Positive Rate', fontsize=11)
    ax.set_title('ROC Curve 비교', fontsize=12, fontweight='bold')
    ax.legend(loc='lower right', fontsize=10)
    ax.grid(True, alpha=0.3)
    
    # 성능 지표 비교
    ax = axes[0, 1]
    metrics = ['R²', 'Accuracy', 'F1-Score', 'AUC']
    encoder_vals = [
        encoder_results['r2'],
        encoder_results['outcome_accuracy'],
        encoder_results['outcome_f1'],
        encoder_results['outcome_auc']
    ]
    autoregressive_vals = [
        autoregressive_results['r2'],
        autoregressive_results['outcome_accuracy'],
        autoregressive_results['outcome_f1'],
        autoregressive_results['outcome_auc']
    ]
    
    x = np.arange(len(metrics))
    width = 0.35
    ax.bar(x - width/2, encoder_vals, width, label='Encoder-only', alpha=0.8)
    ax.bar(x + width/2, autoregressive_vals, width, label='Autoregressive', alpha=0.8)
    ax.set_ylabel('Score', fontsize=11)
    ax.set_title('성능 지표 비교', fontsize=12, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(metrics)
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3, axis='y')
    ax.set_ylim([0, 1])
    
    # Confusion Matrix
    ax = axes[1, 0]
    cm_combined = np.array([
        encoder_results['confusion_matrix'].flatten(),
        autoregressive_results['confusion_matrix'].flatten()
    ])
    im = ax.imshow(cm_combined, cmap='Blues', aspect='auto')
    ax.set_xticks([0, 1, 2, 3])
    ax.set_xticklabels(['TN', 'FP', 'FN', 'TP'])
    ax.set_yticks([0, 1])
    ax.set_yticklabels(['Encoder-only', 'Autoregressive'])
    ax.set_title('Confusion Matrix 비교', fontsize=12, fontweight='bold')
    
    for i in range(2):
        for j in range(4):
            text = ax.text(j, i, f'{cm_combined[i, j]:.0f}',
                          ha="center", va="center", color="w", fontsize=11)
    
    # MAE 비교
    ax = axes[1, 1]
    models = ['Encoder-only', 'Autoregressive']
    mae_vals = [encoder_results['mae'], autoregressive_results['mae']]
    colors = ['#ff7f0e', '#2ca02c']
    bars = ax.bar(models, mae_vals, color=colors, alpha=0.8, edgecolor='black', linewidth=1.5)
    ax.set_ylabel('MAE', fontsize=11)
    ax.set_title('평균 절대 오차 (MAE) 비교', fontsize=12, fontweight='bold')
    ax.grid(True, alpha=0.3, axis='y')
    
    for bar, val in zip(bars, mae_vals):
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
                f'{val:.6f}', ha='center', va='bottom', fontsize=11)
    
    plt.tight_layout()
    plt.savefig(f'{MODEL_DIR}/plots/encoder_auto_improved_comparison.png', dpi=300, bbox_inches='tight')
    print(f"✅ 저장: {MODEL_DIR}/plots/encoder_auto_improved_comparison.png")
    plt.close()

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 메인 함수
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def main():
    """메인 함수"""
    
    global encoder_model, autoregressive_model
    global encoder_results, autoregressive_results
    
    print("\n" + "="*80)
    print("Valorant 라운드 예측: Encoder-only & Autoregressive 개선 버전")
    print("="*80)
    print("\n개선사항:")
    print("1. Encoder-only:")
    print("   - Time-distributed Dense로 각 타임스텝 다양화")
    print("   - 더 큰 LSTM (128-256-128)")
    print("   - Dropout 감소 (0.3 → 0.2)")
    print("   - Batch Size 증가 (16 → 32)")
    print("\n2. Autoregressive:")
    print("   - 진짜 순차 생성 (한 스텝씩 피드백)")
    print("   - 더 큰 LSTM (128-192-128)")
    print("   - Dropout 감소 (0.3 → 0.2)")
    print("="*80)
    
    os.makedirs(MODEL_DIR, exist_ok=True)
    
    # Step 1: 데이터 로드
    train_data, val_data, test_data, metadata = load_and_prepare_data()
    
    # Step 2: 모델 구축 (개선 버전)
    print(f"\n{'='*80}")
    print("모델 구축 (개선 버전)")
    print(f"{'='*80}")
    
    encoder_model = build_encoder_only_improved()
    autoregressive_model = build_autoregressive_improved()
    
    print(f"\n✅ 두 모델 구축 완료")
    
    # Step 3: 모델 학습
    print(f"\n{'='*80}")
    print("모델 학습")
    print(f"{'='*80}")
    
    encoder_history, encoder_metrics_callback = train_model(
        encoder_model, 'encoder_only', train_data, val_data
    )
    
    print(f"\n" + "-"*80 + "\n")
    
    autoregressive_history, autoregressive_metrics_callback = train_model(
        autoregressive_model, 'autoregressive', train_data, val_data
    )
    
    # Step 4: 모델 평가
    print(f"\n{'='*80}")
    print("모델 평가")
    print(f"{'='*80}")
    
    encoder_results = evaluate_model(encoder_model, 'encoder_only', test_data)
    autoregressive_results = evaluate_model(autoregressive_model, 'autoregressive', test_data)
    
    # Step 5: 최종 비교
    print(f"\n{'='*80}")
    print("최종 비교 결과")
    print(f"{'='*80}")
    
    print(f"\n📊 성능 지표 비교 (Test 세트):")
    print(f"\n{'지표':<30} {'Encoder-only':>20} {'Autoregressive':>20}")
    print(f"{'-'*70}")
    
    comparison_data = [
        ('R² Score', encoder_results['r2'], autoregressive_results['r2']),
        ('MAE', encoder_results['mae'], autoregressive_results['mae']),
        ('RMSE', encoder_results['rmse'], autoregressive_results['rmse']),
        ('Outcome Accuracy', encoder_results['outcome_accuracy'], autoregressive_results['outcome_accuracy']),
        ('F1-Score', encoder_results['outcome_f1'], autoregressive_results['outcome_f1']),
        ('AUC-ROC', encoder_results['outcome_auc'], autoregressive_results['outcome_auc']),
    ]
    
    for metric_name, encoder_val, autoregressive_val in comparison_data:
        print(f"{metric_name:<30} {encoder_val:>20.4f} {autoregressive_val:>20.4f}")
    
    # Step 6: 개선 효과 분석
    print(f"\n{'='*80}")
    print("개선 효과 분석 (이전 vs 현재)")
    print(f"{'='*80}")
    
    # 이전 결과 (하드코딩)
    prev_encoder = {'r2': 0.3516, 'auc': 0.6423, 'acc': 0.5965}
    prev_auto = {'r2': 0.1664, 'auc': 0.6278, 'acc': 0.5903}
    
    print(f"\n✅ Encoder-only:")
    print(f"  R²:  {prev_encoder['r2']:.4f} → {encoder_results['r2']:.4f} ({"+" if encoder_results['r2'] > prev_encoder['r2'] else ""}{(encoder_results['r2'] - prev_encoder['r2']):.4f})")
    print(f"  AUC: {prev_encoder['auc']:.4f} → {encoder_results['outcome_auc']:.4f} ({"+" if encoder_results['outcome_auc'] > prev_encoder['auc'] else ""}{(encoder_results['outcome_auc'] - prev_encoder['auc']):.4f})")
    print(f"  Acc: {prev_encoder['acc']:.4f} → {encoder_results['outcome_accuracy']:.4f} ({"+" if encoder_results['outcome_accuracy'] > prev_encoder['acc'] else ""}{(encoder_results['outcome_accuracy'] - prev_encoder['acc']):.4f})")
    
    print(f"\n✅ Autoregressive:")
    print(f"  R²:  {prev_auto['r2']:.4f} → {autoregressive_results['r2']:.4f} ({"+" if autoregressive_results['r2'] > prev_auto['r2'] else ""}{(autoregressive_results['r2'] - prev_auto['r2']):.4f})")
    print(f"  AUC: {prev_auto['auc']:.4f} → {autoregressive_results['outcome_auc']:.4f} ({"+" if autoregressive_results['outcome_auc'] > prev_auto['auc'] else ""}{(autoregressive_results['outcome_auc'] - prev_auto['auc']):.4f})")
    print(f"  Acc: {prev_auto['acc']:.4f} → {autoregressive_results['outcome_accuracy']:.4f} ({"+" if autoregressive_results['outcome_accuracy'] > prev_auto['acc'] else ""}{(autoregressive_results['outcome_accuracy'] - prev_auto['acc']):.4f})")
    
    # Step 7: 시각화
    print(f"\n{'='*80}")
    print("시각화")
    print(f"{'='*80}")
    
    plot_comparison(encoder_results, autoregressive_results)
    
    # Step 8: 최종 저장
    print(f"\n{'='*80}")
    print("최종 결과 저장")
    print(f"{'='*80}")
    
    final_results = {
        'encoder_only_results': encoder_results,
        'autoregressive_results': autoregressive_results,
        'comparison_data': comparison_data,
    }
    
    with open(f'{MODEL_DIR}/improved_results.pkl', 'wb') as f:
        pickle.dump(final_results, f)
    
    print(f"\n✅ 저장 완료:")
    print(f"  - {MODEL_DIR}/encoder_only_model_improved.keras")
    print(f"  - {MODEL_DIR}/autoregressive_model_improved.keras")
    print(f"  - {MODEL_DIR}/improved_results.pkl")
    print(f"  - {MODEL_DIR}/plots/encoder_auto_improved_comparison.png")
    
    print(f"\n{'='*80}")
    print("✅ 전체 파이프라인 완료!")
    print(f"{'='*80}")

if __name__ == "__main__":
    main()

In [ ]:

import numpy as np
import pandas as pd
import os
import pickle
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    mean_squared_error, r2_score, accuracy_score, f1_score,
    roc_curve, auc, roc_auc_score, confusion_matrix, classification_report
)
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model
from tensorflow.keras.callbacks import Callback, EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.optimizers import Adam
import warnings
warnings.filterwarnings('ignore')

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 전역 변수
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

encoder_model = None
autoregressive_model = None

encoder_results = None
autoregressive_results = None

X_train = None
Y_train = None
X_val = None
Y_val = None
X_test = None
Y_test = None

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 설정
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

DATA_DIR = './data/seq2seq_data'
MODEL_DIR = './models/seq2seq_comparison'
BATCH_SIZE = 32  # ✅ 16 → 32
EPOCHS = 160  # ✅ 100 → 160
SEED = 42

SEQUENCE_LENGTH = 12
FUTURE_STEPS = 12
N_INPUT_FEATURES = 8
N_OUTPUT_FEATURES = 9
N_DECODER_INPUT = 8

np.random.seed(SEED)
tf.random.set_seed(SEED)

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# Callback: 상세 지표 계산
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

class MetricsCallback(Callback):
    """매 에포크마다 상세 지표 계산"""

    def __init__(self, X_val, Y_val, model_type='encoder_only'):
        super().__init__()
        self.X_val = X_val
        self.Y_val = Y_val
        self.model_type = model_type
        self.history = {
            'val_r2': [],
            'val_mae': [],
            'val_outcome_acc': [],
            'val_outcome_f1': [],
            'val_auc': [],
        }

    def on_epoch_end(self, epoch, logs=None):
        """에포크 끝날 때마다 호출"""

        if self.model_type == 'encoder_only':
            val_pred = self.model.predict(self.X_val, verbose=0)
        else:  # autoregressive - 진짜 순차 생성
            val_pred = autoregressive_predict_real(self.model, self.X_val)

        val_pred_features = val_pred[:, :, :-1]
        val_targets_features = self.Y_val[:, :, :-1]

        val_pred_outcome_prob = val_pred[:, :, -1].flatten()
        val_pred_outcome = (val_pred_outcome_prob > 0.5).astype(int)
        val_targets_outcome = self.Y_val[:, :, -1].astype(int).flatten()

        try:
            r2_scores = []
            for feat_idx in range(N_DECODER_INPUT):
                r2 = r2_score(
                    val_targets_features[:, :, feat_idx].flatten(),
                    val_pred_features[:, :, feat_idx].flatten()
                )
                r2_scores.append(r2)
            val_r2 = np.mean(r2_scores)
        except:
            val_r2 = -1.0

        val_mae = np.mean(np.abs(val_targets_features - val_pred_features))
        val_outcome_acc = accuracy_score(val_targets_outcome, val_pred_outcome)
        val_outcome_f1 = f1_score(val_targets_outcome, val_pred_outcome, zero_division=0)

        try:
            val_auc = roc_auc_score(val_targets_outcome, val_pred_outcome_prob)
        except:
            val_auc = 0.5

        self.history['val_r2'].append(val_r2)
        self.history['val_mae'].append(val_mae)
        self.history['val_outcome_acc'].append(val_outcome_acc)
        self.history['val_outcome_f1'].append(val_outcome_f1)
        self.history['val_auc'].append(val_auc)

        if (epoch + 1) % 20 == 0:
            print(f"\n[Epoch {epoch+1}] {self.model_type.upper()} Validation:")
            print(f"  R²: {val_r2:.4f}, MAE: {val_mae:.6f}, AUC: {val_auc:.4f}")
            print(f"  Outcome Acc: {val_outcome_acc:.4f}, F1: {val_outcome_f1:.4f}")

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# Autoregressive 진짜 순차 생성 (개선)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def autoregressive_predict_real(model, X_batch, batch_size=32):
    """
    ✅ 진짜 Autoregressive 예측 (한 스텝씩 순차 생성)

    이전 예측을 다음 입력으로 사용 (피드백)
    """

    total_samples = len(X_batch)
    predictions = np.zeros((total_samples, FUTURE_STEPS, N_OUTPUT_FEATURES), dtype=np.float32)

    for batch_start in range(0, total_samples, batch_size):
        batch_end = min(batch_start + batch_size, total_samples)
        X_batch_slice = X_batch[batch_start:batch_end]
        batch_len = len(X_batch_slice)

        # Decoder 입력 초기화 (0으로 시작)
        decoder_seq = np.zeros((batch_len, FUTURE_STEPS, N_DECODER_INPUT), dtype=np.float32)

        # ✅ 한 스텝씩 순차 생성
        for step in range(FUTURE_STEPS):
            # 현재까지의 decoder 입력으로 예측
            pred = model.predict([X_batch_slice, decoder_seq], verbose=0)

            # step번째 라운드 예측 저장
            predictions[batch_start:batch_end, step, :] = pred[:, step, :]

            # ✅ 다음 스텝을 위해 현재 예측을 decoder 입력으로 피드백
            if step < FUTURE_STEPS - 1:
                decoder_seq[:, step + 1, :] = pred[:, step, :N_DECODER_INPUT]

        percent = (batch_end / total_samples) * 100
        print(f"    [{batch_end}/{total_samples}] {percent:.1f}%", end='\r', flush=True)

    print(f"    [{total_samples}/{total_samples}] 100.0%")

    return predictions

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 모델 1: Encoder-only (개선 버전)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def build_encoder_only_improved():
    """
    ✅ 개선된 Encoder-only 모델

    핵심 개선:
    - RepeatVector 후 Time-distributed Dense로 각 타임스텝 다양화
    - 더 큰 LSTM 유닛 (128-256-128)
    - Dropout 감소 (0.3 → 0.2)
    """

    print("\n✅ [Encoder-only] 개선 모델 구축 중...")

    # ━━━━━━━ Encoder ━━━━━━━
    encoder_inputs = keras.Input(shape=(SEQUENCE_LENGTH, N_INPUT_FEATURES), name='encoder_input')

    # Encoder Layer 1
    encoder_layer1 = layers.Bidirectional(
        layers.LSTM(128, return_sequences=True, recurrent_dropout=0.2, dropout=0.2),
        name='encoder_bilstm1'
    )(encoder_inputs)
    encoder_bn1 = layers.BatchNormalization()(encoder_layer1)
    encoder_do1 = layers.Dropout(0.2)(encoder_bn1)

    # Encoder Layer 2
    encoder_layer2 = layers.Bidirectional(
        layers.LSTM(256, return_sequences=True, recurrent_dropout=0.2, dropout=0.2),
        name='encoder_bilstm2'
    )(encoder_do1)
    encoder_bn2 = layers.BatchNormalization()(encoder_layer2)
    encoder_do2 = layers.Dropout(0.2)(encoder_bn2)

    # Encoder Layer 3
    encoder_layer3 = layers.Bidirectional(
        layers.LSTM(128, return_state=True, recurrent_dropout=0.2, dropout=0.2),
        name='encoder_bilstm3'
    )
    encoder_outputs, forward_h, forward_c, backward_h, backward_c = encoder_layer3(encoder_do2)
    encoder_bn3 = layers.BatchNormalization()(encoder_outputs)

    # Context vector (더 큰 용량)
    state_h = layers.Concatenate()([forward_h, backward_h])
    state_c = layers.Concatenate()([forward_c, backward_c])

    context_1 = layers.Concatenate()([state_h, state_c])
    context_dense1 = layers.Dense(512, activation='relu')(context_1)
    context_bn1 = layers.BatchNormalization()(context_dense1)
    context_do1 = layers.Dropout(0.3)(context_bn1)

    context_dense2 = layers.Dense(384, activation='relu')(context_do1)
    context_bn2 = layers.BatchNormalization()(context_dense2)
    context_do2 = layers.Dropout(0.3)(context_bn2)

    context_final = layers.Dense(256, activation='relu')(context_do2)
    context_final_bn = layers.BatchNormalization()(context_final)

    # ━━━━━━━ Decoder (개선: Time-distributed Dense로 다양화) ━━━━━━━
    decoder_repeated = layers.RepeatVector(FUTURE_STEPS)(context_final_bn)

    # ✅ Time-distributed Dense로 각 타임스텝을 다르게 변환
    time_diverse = layers.TimeDistributed(
        layers.Dense(256, activation='tanh'),
        name='time_diversification'
    )(decoder_repeated)
    time_diverse_bn = layers.TimeDistributed(layers.BatchNormalization())(time_diverse)

    # Decoder Layer 1
    decoder_layer1 = layers.LSTM(256, return_sequences=True, recurrent_dropout=0.2, dropout=0.2, name='decoder_lstm1')
    decoder_outputs1 = decoder_layer1(time_diverse_bn)
    decoder_bn1 = layers.BatchNormalization()(decoder_outputs1)
    decoder_do1 = layers.Dropout(0.3)(decoder_bn1)

    # Decoder Layer 2
    decoder_layer2 = layers.LSTM(384, return_sequences=True, recurrent_dropout=0.2, dropout=0.2, name='decoder_lstm2')
    decoder_outputs2 = decoder_layer2(decoder_do1)
    decoder_bn2 = layers.BatchNormalization()(decoder_outputs2)
    decoder_do2 = layers.Dropout(0.3)(decoder_bn2)

    # Decoder Layer 3
    decoder_layer3 = layers.LSTM(256, return_sequences=True, recurrent_dropout=0.2, dropout=0.2, name='decoder_lstm3')
    decoder_outputs3 = decoder_layer3(decoder_do2)
    decoder_bn3 = layers.BatchNormalization()(decoder_outputs3)
    decoder_do3 = layers.Dropout(0.2)(decoder_bn3)

    # Dense layers
    dense1 = layers.TimeDistributed(layers.Dense(192, activation='relu'))(decoder_do3)
    dense1_bn = layers.TimeDistributed(layers.BatchNormalization())(dense1)
    dense1_do = layers.TimeDistributed(layers.Dropout(0.2))(dense1_bn)

    dense2 = layers.TimeDistributed(layers.Dense(96, activation='relu'))(dense1_do)
    dense2_bn = layers.TimeDistributed(layers.BatchNormalization())(dense2)
    dense2_do = layers.TimeDistributed(layers.Dropout(0.2))(dense2_bn)

    dense3 = layers.TimeDistributed(layers.Dense(48, activation='relu'))(dense2_do)
    dense3_bn = layers.TimeDistributed(layers.BatchNormalization())(dense3)

    # Output layer
    outputs = layers.TimeDistributed(layers.Dense(N_OUTPUT_FEATURES))(dense3_bn)

    model = Model(encoder_inputs, outputs, name='EncoderOnly_Improved_Model')

    return model

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 모델 2: Autoregressive (개선 버전 - 상태 맞춤)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def build_autoregressive_improved():
    """
    ✅ 개선된 Autoregressive 모델

    핵심 개선:
    - Decoder LSTM을 256으로 통일 (상태 차원 일치)
    - 더 큰 LSTM 유닛
    - Dropout 감소 (0.3 → 0.2)
    """

    print("\n✅ [Autoregressive] 개선 모델 구축 중...")

    # ━━━━━━━ Encoder ━━━━━━━
    encoder_inputs = keras.Input(shape=(SEQUENCE_LENGTH, N_INPUT_FEATURES), name='encoder_input')

    # Encoder Layer 1
    encoder_layer1 = layers.Bidirectional(
        layers.LSTM(128, return_sequences=True, recurrent_dropout=0.3, dropout=0.3),
        name='encoder_bilstm1'
    )(encoder_inputs)
    encoder_bn1 = layers.BatchNormalization()(encoder_layer1)
    encoder_do1 = layers.Dropout(0.3)(encoder_bn1)

    # Encoder Layer 2
    encoder_layer2 = layers.Bidirectional(
        layers.LSTM(192, return_sequences=True, recurrent_dropout=0.3, dropout=0.3),
        name='encoder_bilstm2'
    )(encoder_do1)
    encoder_bn2 = layers.BatchNormalization()(encoder_layer2)
    encoder_do2 = layers.Dropout(0.3)(encoder_bn2)

    # Encoder Layer 3
    encoder_layer3 = layers.Bidirectional(
        layers.LSTM(128, return_state=True, recurrent_dropout=0.3, dropout=0.3),
        name='encoder_bilstm3'
    )
    encoder_outputs, forward_h, forward_c, backward_h, backward_c = encoder_layer3(encoder_do2)
    encoder_bn3 = layers.BatchNormalization()(encoder_outputs)

    # Context vector (256차원)
    state_h = layers.Concatenate()([forward_h, backward_h])
    state_c = layers.Concatenate()([forward_c, backward_c])

    # ━━━━━━━ Decoder ━━━━━━━
    decoder_inputs = keras.Input(shape=(FUTURE_STEPS, N_DECODER_INPUT), name='decoder_input')

    # ✅ 모든 Decoder LSTM을 256으로 통일 (상태 차원 일치)
    decoder_layer1 = layers.LSTM(256, return_sequences=True, recurrent_dropout=0.3, dropout=0.3, name='decoder_lstm1')
    decoder_outputs1 = decoder_layer1(decoder_inputs, initial_state=[state_h, state_c])
    decoder_bn1 = layers.BatchNormalization()(decoder_outputs1)
    decoder_do1 = layers.Dropout(0.2)(decoder_bn1)

    decoder_layer2 = layers.LSTM(256, return_sequences=True, recurrent_dropout=0.3, dropout=0.3, name='decoder_lstm2')
    decoder_outputs2 = decoder_layer2(decoder_do1)
    decoder_bn2 = layers.BatchNormalization()(decoder_outputs2)
    decoder_do2 = layers.Dropout(0.3)(decoder_bn2)

    decoder_layer3 = layers.LSTM(256, return_sequences=True, recurrent_dropout=0.3, dropout=0.3, name='decoder_lstm3')
    decoder_outputs3 = decoder_layer3(decoder_do2)
    decoder_bn3 = layers.BatchNormalization()(decoder_outputs3)
    decoder_do3 = layers.Dropout(0.3)(decoder_bn3)

    # Dense layers
    dense1 = layers.TimeDistributed(layers.Dense(128, activation='relu'))(decoder_do3)
    dense1_bn = layers.TimeDistributed(layers.BatchNormalization())(dense1)
    dense1_do = layers.TimeDistributed(layers.Dropout(0.3))(dense1_bn)

    dense2 = layers.TimeDistributed(layers.Dense(64, activation='relu'))(dense1_do)
    dense2_bn = layers.TimeDistributed(layers.BatchNormalization())(dense2)
    dense2_do = layers.TimeDistributed(layers.Dropout(0.25))(dense2_bn)

    dense3 = layers.TimeDistributed(layers.Dense(32, activation='relu'))(dense2_do)
    dense3_bn = layers.TimeDistributed(layers.BatchNormalization())(dense3)

    # Output layer
    outputs = layers.TimeDistributed(layers.Dense(N_OUTPUT_FEATURES))(dense3_bn)

    model = Model([encoder_inputs, decoder_inputs], outputs, name='Autoregressive_Improved_Model')

    return model

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 데이터 로드
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def load_and_prepare_data():
    """데이터 로드 및 분할"""

    global X_train, Y_train, X_val, Y_val, X_test, Y_test

    print("\n" + "="*80)
    print("데이터 로드 및 준비")
    print("="*80)

    X = np.load(f'{DATA_DIR}/X_sequences.npy')
    Y = np.load(f'{DATA_DIR}/Y_sequences.npy')

    with open(f'{DATA_DIR}/metadata.pkl', 'rb') as f:
        metadata = pickle.load(f)

    print(f"✅ 데이터 로드: X {X.shape}, Y {Y.shape}")

    indices = np.arange(len(X))
    train_idx, test_idx = train_test_split(indices, test_size=0.2, random_state=SEED)
    train_idx, val_idx = train_test_split(train_idx, test_size=0.2/(1-0.2), random_state=SEED)

    X_train, Y_train = X[train_idx], Y[train_idx]
    X_val, Y_val = X[val_idx], Y[val_idx]
    X_test, Y_test = X[test_idx], Y[test_idx]

    print(f"\n데이터 분할:")
    print(f"  Train: {len(X_train):,}")
    print(f"  Val:   {len(X_val):,}")
    print(f"  Test:  {len(X_test):,}")

    return (X_train, Y_train), (X_val, Y_val), (X_test, Y_test), metadata

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 모델 학습 함수
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def train_model(model, model_type, train_data, val_data):
    """모델 학습 + 저장"""

    global X_train, Y_train, X_val, Y_val

    print(f"\n{'='*80}")
    print(f"모델 학습: {model_type.upper()} (개선 버전)")
    print(f"{'='*80}")

    X_train_local, Y_train_local = train_data
    X_val_local, Y_val_local = val_data

    print(f"\n[1/3] 모델 컴파일 중...")
    optimizer = Adam(learning_rate=0.0003, clipnorm=1.0)
    model.compile(optimizer=optimizer, loss='mse', metrics=['mae'])
    print(f"✅ 컴파일 완료")
    print(f"   총 파라미터: {model.count_params():,}")

    print(f"\n[2/3] Callback 설정 중...")
    metrics_callback = MetricsCallback(X_val_local, Y_val_local, model_type=model_type)

    callbacks = [
        EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True, verbose=1),
        ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-7, verbose=1),
        ModelCheckpoint(f'{MODEL_DIR}/best_{model_type}_improved.keras', monitor='val_loss', save_best_only=True, verbose=0),
        metrics_callback
    ]
    print(f"✅ Callback 설정 완료")

    print(f"\n[3/3] 모델 학습 중...")

    if model_type == 'autoregressive':
        decoder_inputs_train = Y_train_local[:, :, :-1]
        decoder_inputs_val = Y_val_local[:, :, :-1]

        history = model.fit(
            [X_train_local, decoder_inputs_train],
            Y_train_local,
            batch_size=BATCH_SIZE,
            epochs=EPOCHS,
            validation_data=([X_val_local, decoder_inputs_val], Y_val_local),
            callbacks=callbacks,
            verbose=1
        )
    else:  # encoder_only
        history = model.fit(
            X_train_local,
            Y_train_local,
            batch_size=BATCH_SIZE,
            epochs=EPOCHS,
            validation_data=(X_val_local, Y_val_local),
            callbacks=callbacks,
            verbose=1
        )

    print(f"\n✅ 학습 완료")

    print(f"\n[4/3] 모델 저장 중 (.keras 포맷)...", end='', flush=True)
    model.save(f'{MODEL_DIR}/{model_type}_model_improved.keras', save_format='keras')
    print(" ✅")
    print(f"  저장 위치: {MODEL_DIR}/{model_type}_model_improved.keras")

    return history, metrics_callback

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 모델 평가 함수
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def evaluate_model(model, model_type, test_data):
    """모델 평가"""

    print(f"\n{'='*80}")
    print(f"모델 평가: {model_type.upper()}")
    print(f"{'='*80}")

    X_test_local, Y_test_local = test_data
    total_samples = len(X_test_local)

    print(f"\n[1/2] 예측 중 [0/{total_samples}]...", end='\r', flush=True)

    if model_type == 'encoder_only':
        predictions = model.predict(X_test_local, batch_size=32, verbose=0)
        print(f"[1/2] 예측 완료 [{total_samples}/{total_samples}] ✅")
    else:  # autoregressive - 진짜 순차 생성
        print(f"\n[1/2] Autoregressive 예측 중 (순차 생성)...")
        predictions = autoregressive_predict_real(model, X_test_local, batch_size=32)

    print(f"\n[2/2] 평가 중...", end='', flush=True)

    pred_features = predictions[:, :, :-1]
    target_features = Y_test_local[:, :, :-1]

    pred_outcome_prob = predictions[:, :, -1].flatten()
    pred_outcome = (pred_outcome_prob > 0.5).astype(int)
    target_outcome = Y_test_local[:, :, -1].astype(int).flatten()

    r2_scores = []
    for feat_idx in range(N_DECODER_INPUT):
        r2 = r2_score(
            target_features[:, :, feat_idx].flatten(),
            pred_features[:, :, feat_idx].flatten()
        )
        r2_scores.append(r2)

    r2 = np.mean(r2_scores)
    mae = np.mean(np.abs(target_features - pred_features))
    mse = np.mean((target_features - pred_features) ** 2)
    rmse = np.sqrt(mse)

    outcome_acc = accuracy_score(target_outcome, pred_outcome)
    outcome_f1 = f1_score(target_outcome, pred_outcome, zero_division=0)
    outcome_auc = roc_auc_score(target_outcome, pred_outcome_prob)

    fpr, tpr, thresholds = roc_curve(target_outcome, pred_outcome_prob)
    cm = confusion_matrix(target_outcome, pred_outcome)

    print(" ✅")

    print(f"\n📊 성능 지표 (Test):")
    print(f"  R² Score: {r2:.4f}")
    print(f"  MAE: {mae:.6f}")
    print(f"  RMSE: {rmse:.6f}")
    print(f"  Outcome Accuracy: {outcome_acc:.4f} ({outcome_acc*100:.2f}%)")
    print(f"  Outcome F1-Score: {outcome_f1:.4f}")
    print(f"  Outcome AUC-ROC: {outcome_auc:.4f}")

    print(f"\n📈 Confusion Matrix:")
    print(f"  True Negatives: {cm[0, 0]}, False Positives: {cm[0, 1]}")
    print(f"  False Negatives: {cm[1, 0]}, True Positives: {cm[1, 1]}")

    return {
        'r2': r2,
        'mae': mae,
        'mse': mse,
        'rmse': rmse,
        'outcome_accuracy': outcome_acc,
        'outcome_f1': outcome_f1,
        'outcome_auc': outcome_auc,
        'fpr': fpr,
        'tpr': tpr,
        'threshold': thresholds,
        'pred_outcome_prob': pred_outcome_prob,
        'target_outcome': target_outcome,
        'confusion_matrix': cm,
    }

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 시각화 함수
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def plot_comparison(encoder_results, autoregressive_results):
    """두 모델 비교 시각화"""

    print(f"\n{'='*80}")
    print("시각화: Encoder-only vs Autoregressive (개선 버전)")
    print(f"{'='*80}")

    os.makedirs(f'{MODEL_DIR}/plots', exist_ok=True)

    fig, axes = plt.subplots(2, 2, figsize=(15, 12))
    fig.suptitle('Encoder-only vs Autoregressive (개선 버전, EPOCHS=160)', fontsize=16, fontweight='bold')

    # ROC Curve
    ax = axes[0, 0]
    ax.plot([0, 1], [0, 1], 'k--', label='Random Classifier', linewidth=2)
    ax.plot(encoder_results['fpr'], encoder_results['tpr'],
            label=f"Encoder-only (AUC = {encoder_results['outcome_auc']:.4f})",
            linewidth=2.5, marker='s', markersize=4)
    ax.plot(autoregressive_results['fpr'], autoregressive_results['tpr'],
            label=f"Autoregressive (AUC = {autoregressive_results['outcome_auc']:.4f})",
            linewidth=2.5, marker='^', markersize=4)
    ax.set_xlabel('False Positive Rate', fontsize=11)
    ax.set_ylabel('True Positive Rate', fontsize=11)
    ax.set_title('ROC Curve 비교', fontsize=12, fontweight='bold')
    ax.legend(loc='lower right', fontsize=10)
    ax.grid(True, alpha=0.3)

    # 성능 지표 비교
    ax = axes[0, 1]
    metrics = ['R²', 'Accuracy', 'F1-Score', 'AUC']
    encoder_vals = [
        encoder_results['r2'],
        encoder_results['outcome_accuracy'],
        encoder_results['outcome_f1'],
        encoder_results['outcome_auc']
    ]
    autoregressive_vals = [
        autoregressive_results['r2'],
        autoregressive_results['outcome_accuracy'],
        autoregressive_results['outcome_f1'],
        autoregressive_results['outcome_auc']
    ]

    x = np.arange(len(metrics))
    width = 0.35
    ax.bar(x - width/2, encoder_vals, width, label='Encoder-only', alpha=0.8)
    ax.bar(x + width/2, autoregressive_vals, width, label='Autoregressive', alpha=0.8)
    ax.set_ylabel('Score', fontsize=11)
    ax.set_title('성능 지표 비교', fontsize=12, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(metrics)
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3, axis='y')
    ax.set_ylim([0, 1])

    # Confusion Matrix
    ax = axes[1, 0]
    cm_combined = np.array([
        encoder_results['confusion_matrix'].flatten(),
        autoregressive_results['confusion_matrix'].flatten()
    ])
    im = ax.imshow(cm_combined, cmap='Blues', aspect='auto')
    ax.set_xticks([0, 1, 2, 3])
    ax.set_xticklabels(['TN', 'FP', 'FN', 'TP'])
    ax.set_yticks([0, 1])
    ax.set_yticklabels(['Encoder-only', 'Autoregressive'])
    ax.set_title('Confusion Matrix 비교', fontsize=12, fontweight='bold')

    for i in range(2):
        for j in range(4):
            text = ax.text(j, i, f'{cm_combined[i, j]:.0f}',
                          ha="center", va="center", color="w", fontsize=11)

    # MAE 비교
    ax = axes[1, 1]
    models = ['Encoder-only', 'Autoregressive']
    mae_vals = [encoder_results['mae'], autoregressive_results['mae']]
    colors = ['#ff7f0e', '#2ca02c']
    bars = ax.bar(models, mae_vals, color=colors, alpha=0.8, edgecolor='black', linewidth=1.5)
    ax.set_ylabel('MAE', fontsize=11)
    ax.set_title('평균 절대 오차 (MAE) 비교', fontsize=12, fontweight='bold')
    ax.grid(True, alpha=0.3, axis='y')

    for bar, val in zip(bars, mae_vals):
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
                f'{val:.6f}', ha='center', va='bottom', fontsize=11)

    plt.tight_layout()
    plt.savefig(f'{MODEL_DIR}/plots/encoder_auto_improved_comparison.png', dpi=300, bbox_inches='tight')
    print(f"✅ 저장: {MODEL_DIR}/plots/encoder_auto_improved_comparison.png")
    plt.close()

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 메인 함수
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def main():
    """메인 함수"""

    global encoder_model, autoregressive_model
    global encoder_results, autoregressive_results

    print("\n" + "="*80)
    print("Valorant 라운드 예측: Encoder-only & Autoregressive 개선 버전")
    print("="*80)
    print("\n개선사항:")
    print("1. Encoder-only:")
    print("   - Time-distributed Dense로 각 타임스텝 다양화")
    print("   - 더 큰 LSTM (128-256-128)")
    print("   - Dropout 감소 (0.3 → 0.2)")
    print("   - Batch Size: 32")
    print("\n2. Autoregressive:")
    print("   - Decoder LSTM 256으로 통일 (상태 차원 일치)")
    print("   - 진짜 순차 생성 (한 스텝씩 피드백)")
    print("   - Dropout 감소 (0.3 → 0.2)")
    print("\n3. 일반:")
    print(f"   - EPOCHS: 100 → 160")
    print(f"   - Learning Rate: 0.0003")
    print("="*80)

    os.makedirs(MODEL_DIR, exist_ok=True)

    # Step 1: 데이터 로드
    train_data, val_data, test_data, metadata = load_and_prepare_data()

    # Step 2: 모델 구축
    print(f"\n{'='*80}")
    print("모델 구축 (개선 버전)")
    print(f"{'='*80}")

    encoder_model = build_encoder_only_improved()
    autoregressive_model = build_autoregressive_improved()

    print(f"\n✅ 두 모델 구축 완료")

    # Step 3: 모델 학습
    print(f"\n{'='*80}")
    print("모델 학습")
    print(f"{'='*80}")

    encoder_history, encoder_metrics_callback = train_model(
        encoder_model, 'encoder_only', train_data, val_data
    )

    print(f"\n" + "-"*80 + "\n")

    autoregressive_history, autoregressive_metrics_callback = train_model(
        autoregressive_model, 'autoregressive', train_data, val_data
    )

    # Step 4: 모델 평가
    print(f"\n{'='*80}")
    print("모델 평가")
    print(f"{'='*80}")

    encoder_results = evaluate_model(encoder_model, 'encoder_only', test_data)
    autoregressive_results = evaluate_model(autoregressive_model, 'autoregressive', test_data)

    # Step 5: 최종 비교
    print(f"\n{'='*80}")
    print("최종 비교 결과")
    print(f"{'='*80}")

    print(f"\n📊 성능 지표 비교 (Test 세트):")
    print(f"\n{'지표':<30} {'Encoder-only':>20} {'Autoregressive':>20}")
    print(f"{'-'*70}")

    comparison_data = [
        ('R² Score', encoder_results['r2'], autoregressive_results['r2']),
        ('MAE', encoder_results['mae'], autoregressive_results['mae']),
        ('RMSE', encoder_results['rmse'], autoregressive_results['rmse']),
        ('Outcome Accuracy', encoder_results['outcome_accuracy'], autoregressive_results['outcome_accuracy']),
        ('F1-Score', encoder_results['outcome_f1'], autoregressive_results['outcome_f1']),
        ('AUC-ROC', encoder_results['outcome_auc'], autoregressive_results['outcome_auc']),
    ]

    for metric_name, encoder_val, autoregressive_val in comparison_data:
        print(f"{metric_name:<30} {encoder_val:>20.4f} {autoregressive_val:>20.4f}")

    # Step 6: 개선 효과 분석
    print(f"\n{'='*80}")
    print("개선 효과 분석 (이전 vs 현재)")
    print(f"{'='*80}")

    # 이전 결과
    prev_encoder = {'r2': 0.3516, 'auc': 0.6423, 'acc': 0.5965}
    prev_auto = {'r2': 0.1664, 'auc': 0.6278, 'acc': 0.5903}

    print(f"\n✅ Encoder-only:")
    enc_r2_diff = encoder_results['r2'] - prev_encoder['r2']
    enc_auc_diff = encoder_results['outcome_auc'] - prev_encoder['auc']
    enc_acc_diff = encoder_results['outcome_accuracy'] - prev_encoder['acc']
    print(f"  R²:  {prev_encoder['r2']:.4f} → {encoder_results['r2']:.4f} ({'+' if enc_r2_diff >= 0 else ''}{enc_r2_diff:.4f})")
    print(f"  AUC: {prev_encoder['auc']:.4f} → {encoder_results['outcome_auc']:.4f} ({'+' if enc_auc_diff >= 0 else ''}{enc_auc_diff:.4f})")
    print(f"  Acc: {prev_encoder['acc']:.4f} → {encoder_results['outcome_accuracy']:.4f} ({'+' if enc_acc_diff >= 0 else ''}{enc_acc_diff:.4f})")

    print(f"\n✅ Autoregressive:")
    auto_r2_diff = autoregressive_results['r2'] - prev_auto['r2']
    auto_auc_diff = autoregressive_results['outcome_auc'] - prev_auto['auc']
    auto_acc_diff = autoregressive_results['outcome_accuracy'] - prev_auto['acc']
    print(f"  R²:  {prev_auto['r2']:.4f} → {autoregressive_results['r2']:.4f} ({'+' if auto_r2_diff >= 0 else ''}{auto_r2_diff:.4f})")
    print(f"  AUC: {prev_auto['auc']:.4f} → {autoregressive_results['outcome_auc']:.4f} ({'+' if auto_auc_diff >= 0 else ''}{auto_auc_diff:.4f})")
    print(f"  Acc: {prev_auto['acc']:.4f} → {autoregressive_results['outcome_accuracy']:.4f} ({'+' if auto_acc_diff >= 0 else ''}{auto_acc_diff:.4f})")

    # Step 7: 시각화
    print(f"\n{'='*80}")
    print("시각화")
    print(f"{'='*80}")

    plot_comparison(encoder_results, autoregressive_results)

    # Step 8: 최종 저장
    print(f"\n{'='*80}")
    print("최종 결과 저장")
    print(f"{'='*80}")

    final_results = {
        'encoder_only_results': encoder_results,
        'autoregressive_results': autoregressive_results,
        'comparison_data': comparison_data,
    }

    with open(f'{MODEL_DIR}/improved_results.pkl', 'wb') as f:
        pickle.dump(final_results, f)

    print(f"\n✅ 저장 완료:")
    print(f"  - {MODEL_DIR}/encoder_only_model_improved.keras")
    print(f"  - {MODEL_DIR}/autoregressive_model_improved.keras")
    print(f"  - {MODEL_DIR}/improved_results.pkl")
    print(f"  - {MODEL_DIR}/plots/encoder_auto_improved_comparison.png")

    print(f"\n{'='*80}")
    print("✅ 전체 파이프라인 완료! (EPOCHS=160)")
    print(f"{'='*80}")

if __name__ == "__main__":
    main()